# Fixed-Expiry SVI Implied Volatility Calibration

This notebook fits SVI volatility smiles to SPX options data for one fixed expiration observed across multiple dates. The goal is to create a smooth and temporally stable representation of the implied volatility smile.

Raw Bloomberg data are not included in this public repository. The notebook expects local data files in the `data/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize, least_squares
from sklearn.metrics import mean_squared_error

In [ ]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    raise RuntimeError(
        "Could not identify the project root. "
        "Run this notebook from either the project root or the notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root detected.")
print("Data folder:", DATA_DIR.relative_to(PROJECT_ROOT))
print("Output folder:", OUTPUT_DIR.relative_to(PROJECT_ROOT))

assert DATA_DIR.exists(), f"Data folder not found: {DATA_DIR}"

In [ ]:
futures_path = DATA_DIR / "spx_futures.csv"
options_path = DATA_DIR / "options_data.csv"

missing_files = [
    path.name
    for path in [futures_path, options_path]
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Raw Bloomberg data files are not included in this public repository. "
        f"Missing files: {missing_files}. "
        "To rerun the notebook, place the required local data files in the data/ folder."
    )

futures_raw = pd.read_csv(futures_path)
options_raw = pd.read_csv(options_path)

print("Futures shape:", futures_raw.shape) #11629 rows x 11 columns
print("Options shape:", options_raw.shape) #44234 rows x 5 columns

display(futures_raw.head())
display(options_raw.head())

print("Futures columns:")
for column in futures_raw.columns:
    print(repr(column)) 
#query_date, ID, fut_last_trade_dt, px_last, px_bid, px_mid, px_low, px_high, px_open, px_volume, px_ask

print("\nOptions columns:")
for column in options_raw.columns:
    print(repr(column))
#pricedate, expiry, strike, option_type, impliedvol

print("Futures information:")
futures_raw.info() #first three str, rest float64

print("\nOptions information:")
options_raw.info() #price, expiry, option_type: str; strike: int64; impliedvol: float64

def missingness_summary(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Return missing counts and percentages by column."""

    summary = pd.DataFrame(
        {
            "dtype": frame.dtypes.astype(str),
            "missing_count": frame.isna().sum(),
            "missing_percent": (
                frame.isna().mean() * 100
            ),
            "unique_values": frame.nunique(
                dropna=True
            ),
        }
    )

    return summary.sort_values(
        "missing_percent",
        ascending=False,
    )

futures_missingness = missingness_summary(
    futures_raw
)

options_missingness = missingness_summary(
    options_raw
)

print("Futures missingness:")
display(futures_missingness)
#no missingness for str columns, rest mostly around 90% missingness

print("Options missingness:")
display(options_missingness)
#4.6% missingness for impliedvol

plt.figure(figsize=(10, 5))

futures_missingness[
    "missing_percent"
].sort_values().plot(kind="barh")

plt.xlabel("Missing values (%)")
plt.ylabel("Column")
plt.title("Futures Data Missingness")

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))

options_missingness[
    "missing_percent"
].sort_values().plot(kind="barh")

plt.xlabel("Missing values (%)")
plt.ylabel("Column")
plt.title("Options Data Missingness")

plt.tight_layout()
plt.show()

In [ ]:
#working copies
futures = futures_raw.copy()
options = options_raw.copy()

futures.columns = (
    futures.columns
    .astype(str)
    .str.strip()
)

options.columns = (
    options.columns
    .astype(str)
    .str.strip()
)

futures = futures.rename(
    columns={
        "query_date": "query_date",
        "ID": "contract_id",
        "fut_last_trade_dt()": "futures_expiry",
        "px_last(dates=#dt)": "futures_last",
        "px_bid(dates=#dt)": "futures_bid",
        "px_mid(dates=#dt)": "futures_mid",
        "px_low(dates=#dt)": "futures_low",
        "px_high(dates=#dt)": "futures_high",
        "px_open(dates=#dt)": "futures_open",
        "px_volume(dates=#dt)": "futures_volume",
        "px_ask(dates=#dt)": "futures_ask",
    }
)

options = options.rename(
    columns={
        "pricedate": "price_date",
        "Expiry": "option_expiry",
        "strike": "strike",
        "option_type": "option_type",
        "impliedvol": "implied_vol",
    }
)

print("Futures columns:")
print(futures.columns.tolist())

print("\nOptions columns:")
print(options.columns.tolist())

required_futures_columns = {
    "query_date",
    "contract_id",
    "futures_expiry",
    "futures_last",
}

required_options_columns = {
    "price_date",
    "option_expiry",
    "strike",
    "option_type",
    "implied_vol",
}

missing_futures_columns = (
    required_futures_columns
    - set(futures.columns)
)

missing_options_columns = (
    required_options_columns
    - set(options.columns)
)

print(
    "Missing futures columns:",
    missing_futures_columns,
)

print(
    "Missing options columns:",
    missing_options_columns,
)

assert not missing_futures_columns
assert not missing_options_columns

In [ ]:
#standardize the date format across both files
futures["query_date"] = pd.to_datetime(
    futures["query_date"],
    errors="coerce",
)

futures["futures_expiry"] = pd.to_datetime(
    futures["futures_expiry"],
    errors="coerce",
)

options["price_date"] = pd.to_datetime(
    options["price_date"],
    errors="coerce",
    dayfirst=True,
)

options["option_expiry"] = pd.to_datetime(
    options["option_expiry"],
    errors="coerce",
    dayfirst=True,
)

futures_numeric_columns = [
    "futures_last",
    "futures_bid",
    "futures_mid",
    "futures_low",
    "futures_high",
    "futures_open",
    "futures_volume",
    "futures_ask",
]

for column in futures_numeric_columns:
    futures[column] = pd.to_numeric(
        futures[column],
        errors="coerce",
    )

options["strike"] = pd.to_numeric(
    options["strike"],
    errors="coerce",
)

options["implied_vol"] = pd.to_numeric(
    options["implied_vol"],
    errors="coerce",
)

#standardize put and call labels for both files
options["option_type"] = (
    options["option_type"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace(
        {
            "CALL": "C",
            "PUT": "P",
        }
    )
)

print("Futures dtypes:")
display(
    futures.dtypes.to_frame("dtype")
)

print("Options dtypes:")
display(
    options.dtypes.to_frame("dtype")
)

In [ ]:
#basic EDA (options)
print(
    "Option observation-date range:",
    options["price_date"].min(),
    "to",
    options["price_date"].max(),
)

print(
    "Option expiry-date range:",
    options["option_expiry"].min(),
    "to",
    options["option_expiry"].max(),
)

print(
    "Number of observation dates:",
    options["price_date"].nunique(),
)

print(
    "Number of option expiries:",
    options["option_expiry"].nunique(),
)

option_expiry_counts = (
    options["option_expiry"]
    .value_counts()
    .sort_index()
    .rename_axis("option_expiry")
    .to_frame("n_rows")
)

display(option_expiry_counts)

option_expiry_counts = (
    options["option_expiry"]
    .value_counts()
    .sort_index()
    .rename_axis("option_expiry")
    .to_frame("n_rows")
)

display(option_expiry_counts)

display(
    options["option_type"]
    .value_counts(dropna=False)
    .to_frame("n_rows")
)

display(
    options["implied_vol"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

plt.figure(figsize=(9, 5))

plt.hist(
    options["implied_vol"].dropna(),
    bins=60,
)

plt.xlabel("Implied volatility as stored")
plt.ylabel("Number of options")
plt.title("Raw Implied-Volatility Distribution")

plt.tight_layout()
plt.show()

#103 daily snapshots (each has a SVI curve)
#all expire on the same date: 2026-12-18
#calls and puts perfectly balanced
#IV stored in percentage points
#mostly around 12-35% IV, heavy right skew

In [ ]:
#basic futures EDA
print(
    "Futures observation-date range:",
    futures["query_date"].min(),
    "to",
    futures["query_date"].max(),
)

print(
    "Futures expiry-date range:",
    futures["futures_expiry"].min(),
    "to",
    futures["futures_expiry"].max(),
)

print(
    "Number of futures observation dates:",
    futures["query_date"].nunique(),
)

print(
    "Number of futures contracts:",
    futures["contract_id"].nunique(),
)

print(
    "Number of futures expiries:",
    futures["futures_expiry"].nunique(),
)

display(
    futures["contract_id"]
    .value_counts()
    .head(20)
    .to_frame("n_rows")
)

#should be one row per calendar day
#relevant contract is probably #ESZ26 since Z is December


In [ ]:
target_expiry = pd.Timestamp("2026-12-18")

target_futures_raw = futures.loc[
    futures["futures_expiry"] == target_expiry
].copy()

print("Rows with the target expiry:", len(target_futures_raw))

display(
    target_futures_raw["contract_id"]
    .value_counts()
    .to_frame("n_rows")
)

display(
    target_futures_raw[
        [
            "query_date",
            "contract_id",
            "futures_expiry",
            "futures_last",
            "futures_mid",
            "futures_bid",
            "futures_ask",
        ]
    ].head(20)
)

print(
    "Contracts for target expiry:",
    target_futures_raw["contract_id"].nunique(),
)

print(
    "Observation dates for target expiry:",
    target_futures_raw["query_date"].nunique(),
)

#Expiration date given by four different contracts:
#ESZ26 Index
#TSYZ26 Index
#WSPZ26 Index
#HWAZ26 Index
#This one (ESZ26 Index) is still probably right since it's SPX, but need to filter

In [ ]:
TARGET_EXPIRY = pd.Timestamp("2026-12-18")
TARGET_CONTRACT = "ESZ26 Index"

futures_target = (
    futures.loc[
        (futures["futures_expiry"] == TARGET_EXPIRY)
        & (futures["contract_id"] == TARGET_CONTRACT)
    ]
    .copy()
    .sort_values("query_date")
    .reset_index(drop=True)
)

print("Target futures rows:", len(futures_target))
print(
    "Target futures observation dates:",
    futures_target["query_date"].nunique(),
)

display(futures_target.head())

option_dates = (
    options["price_date"]
    .dropna()
    .drop_duplicates()
)

futures_target_on_option_dates = (
    futures_target.loc[
        futures_target["query_date"].isin(option_dates)
    ]
    .copy()
)

print(
    "Target futures rows on option dates:",
    len(futures_target_on_option_dates),
)

print(
    "Unique matched dates:",
    futures_target_on_option_dates[
        "query_date"
    ].nunique(),
)

futures_target_on_option_dates[
    "calculated_mid"
] = (
    futures_target_on_option_dates["futures_bid"]
    + futures_target_on_option_dates["futures_ask"]
) / 2

futures_target_on_option_dates[
    "forward_price"
] = (
    futures_target_on_option_dates["futures_mid"]
    .fillna(
        futures_target_on_option_dates[
            "calculated_mid"
        ]
    )
    .fillna(
        futures_target_on_option_dates[
            "futures_last"
        ]
    )
)

futures_target_on_option_dates[
    "forward_source"
] = np.select(
    [
        futures_target_on_option_dates[
            "futures_mid"
        ].notna(),
        futures_target_on_option_dates[
            "calculated_mid"
        ].notna(),
        futures_target_on_option_dates[
            "futures_last"
        ].notna(),
    ],
    [
        "reported_mid",
        "calculated_mid",
        "last_price",
    ],
    default="missing",
)

display(
    futures_target_on_option_dates[
        "forward_source"
    ].value_counts(dropna=False)
)

missing_target_forwards = (
    futures_target_on_option_dates.loc[
        futures_target_on_option_dates[
            "forward_price"
        ].isna()
    ]
)

print(
    "Option dates without a usable ESZ26 price:",
    len(missing_target_forwards),
)

display(missing_target_forwards.head())

Section 1 - EDA + Data Cleaning
- drop all options with missing implied_vol
- convert implied_vol from a percentage to a decimal (/100)
- if put & call with same strike price (K), observation date (t), and expiration/maturity, choose the one that is out of the money or average IV predictions later
- for each option, find the corresponding future w/ the same observation/query and expiration date (K) (add to table as a new column)
- for each option, calculate time until expiration (days) (T) as a new column
- for each option, calculate total implied variance from implied_vol (= T * (implied_vol)^2) (w) and add as a new column
- for each option, calculate log-moneyness (= log(K/strike price)) (k) and add as a new column
- sort all rows in order of increasing observation date, then within, sort in order of increasing k.

In [ ]:
TARGET_CONTRACT = "ESZ26 Index"

#remove invalid values
option_validity_summary = pd.Series(
    {
        "missing_price_date": options["price_date"].isna().sum(),
        "missing_option_expiry": options["option_expiry"].isna().sum(),
        "missing_strike": options["strike"].isna().sum(),
        "missing_option_type": options["option_type"].isna().sum(),
        "missing_implied_vol": options["implied_vol"].isna().sum(),
        "invalid_option_type": (
            ~options["option_type"].isin(["C", "P"])
        ).sum(),
        "nonpositive_strike": (
            options["strike"].le(0)
        ).sum(),
        "nonpositive_implied_vol": (
            options["implied_vol"].le(0)
        ).sum(),
        "expiry_not_after_observation": (
            options["option_expiry"]
            <= options["price_date"]
        ).sum(),
    },
    name="n_rows",
)

display(option_validity_summary.to_frame())

valid_options_mask = (
    options["price_date"].notna()
    & options["option_expiry"].notna()
    & options["strike"].notna()
    & options["option_type"].isin(["C", "P"])
    & options["implied_vol"].notna()
    & (options["strike"] > 0)
    & (options["implied_vol"] > 0)
    & (
        options["option_expiry"]
        > options["price_date"]
    )
)

options_clean = (
    options.loc[valid_options_mask]
    .copy()
    .reset_index(drop=True)
)

options_removed = options.loc[
    ~valid_options_mask
].copy()

print(
    f"Options retained: "
    f"{len(options_clean):,} / {len(options):,}"
)

print("Options removed:", len(options_removed))

In [ ]:
#convert to decimals
options_clean["iv_decimal"] = (
    options_clean["implied_vol"] / 100
)

display(
    options_clean[
        ["implied_vol", "iv_decimal"]
    ].head()
)

display(
    options_clean["iv_decimal"].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

In [ ]:
#check duplicate option observations
option_key = [
    "price_date",
    "option_expiry",
    "strike",
    "option_type",
]

duplicate_option_mask = options_clean.duplicated(
    subset=option_key,
    keep=False,
)

print(
    "Rows involved in duplicate option keys:",
    duplicate_option_mask.sum(),
)

if duplicate_option_mask.any():
    display(
        options_clean.loc[
            duplicate_option_mask
        ].sort_values(option_key).head(20)
    )

#no duplicates! yay

In [ ]:
#check the correct futures contract
TARGET_EXPIRY = pd.Timestamp("2026-12-18")
TARGET_CONTRACT = "ESZ26 Index"

futures_target = (
    futures.loc[
        (futures["futures_expiry"] == TARGET_EXPIRY)
        & (
            futures["contract_id"]
            == TARGET_CONTRACT
        )
    ]
    .copy()
    .sort_values("query_date")
    .reset_index(drop=True)
)

print("Target futures rows:", len(futures_target))
print(
    "Target futures observation dates:",
    futures_target["query_date"].nunique(),
)

display(futures_target.head())

In [ ]:
#construct the forward price proxy
futures_target["calculated_mid"] = (
    futures_target["futures_bid"]
    + futures_target["futures_ask"]
) / 2

futures_target["forward_price"] = (
    futures_target["futures_mid"]
    .fillna(
        futures_target["calculated_mid"]
    )
    .fillna(
        futures_target["futures_last"]
    )
)

futures_target["forward_source"] = np.select(
    [
        futures_target["futures_mid"].notna(),
        futures_target["calculated_mid"].notna(),
        futures_target["futures_last"].notna(),
    ],
    [
        "reported_mid",
        "calculated_mid",
        "last_price",
    ],
    default="missing",
)

display(
    futures_target["forward_source"]
    .value_counts(dropna=False)
    .to_frame("n_rows")
)
#34 rows we had reported mid, 70 we had last_price, 47 were missing
#but for all 103 rows relevant to the options, there was a price to use.

In [ ]:
option_dates = (
    options_clean["price_date"]
    .drop_duplicates()
    .sort_values()
)

futures_on_option_dates = (
    futures_target.loc[
        futures_target["query_date"].isin(option_dates)
    ]
    .copy()
    .sort_values("query_date")
    .reset_index(drop=True)
)

print(
    "Number of option observation dates:",
    option_dates.nunique(),
)

print(
    "Futures rows on option dates:",
    len(futures_on_option_dates),
)

print(
    "Unique futures dates on option dates:",
    futures_on_option_dates["query_date"].nunique(),
)

In [ ]:
#one and exactly one future value for each option
missing_futures_dates = (
    pd.Index(option_dates)
    .difference(
        futures_on_option_dates["query_date"]
    )
)

print(
    "Option dates without a futures row:",
    len(missing_futures_dates),
)

print(
    "Option dates without a usable forward:",
    futures_on_option_dates[
        "forward_price"
    ].isna().sum(),
)

futures_rows_per_date = (
    futures_on_option_dates
    .groupby("query_date")
    .agg(
        n_rows=("contract_id", "size"),
        n_contracts=("contract_id", "nunique"),
        n_forward_prices=(
            "forward_price",
            "nunique",
        ),
    )
)

display(futures_rows_per_date.head())

assert len(missing_futures_dates) == 0
assert futures_on_option_dates["forward_price"].notna().all()
assert (futures_rows_per_date["n_rows"] == 1).all()
assert (futures_rows_per_date["n_contracts"] == 1).all()
assert (
    futures_rows_per_date["n_forward_prices"] == 1
).all()

print("All option dates have one usable ESZ26 forward.")

In [ ]:
display(
    futures_on_option_dates[
        "forward_source"
    ]
    .value_counts(dropna=False)
    .to_frame("n_rows")
)
#33 used actual reported_mid, 70 used the last price. 

In [ ]:
#rename futures date column to match options date column
forward_by_date = (
    futures_on_option_dates[
        [
            "query_date",
            "futures_expiry",
            "contract_id",
            "forward_price",
            "forward_source",
        ]
    ]
    .rename(
        columns={
            "query_date": "price_date",
            "futures_expiry": "option_expiry",
        }
    )
    .reset_index(drop=True)
)

display(forward_by_date.head())

forward_key = [
    "price_date",
    "option_expiry",
]

assert not forward_by_date.duplicated(
    subset=forward_key
).any()

print("Forward table has one row per date and expiry.")

#merge each option with corresponding future
model_all_quotes = options_clean.merge(
    forward_by_date,
    on=[
        "price_date",
        "option_expiry",
    ],
    how="left",
    validate="many_to_one",
    indicator=True,
)

display(
    model_all_quotes["_merge"]
    .value_counts()
    .to_frame("n_rows")
)

unmatched_options = model_all_quotes.loc[
    model_all_quotes["_merge"] != "both"
].copy()

print(
    "Unmatched option observations:",
    len(unmatched_options),
)

assert unmatched_options.empty

model_all_quotes = (
    model_all_quotes
    .drop(columns="_merge")
    .copy()
)

display(model_all_quotes.head())

In [ ]:
#calculate time to expiration (T)
model_all_quotes["days_to_expiry"] = (
    model_all_quotes["option_expiry"]
    - model_all_quotes["price_date"]
).dt.days

model_all_quotes["time_to_expiry"] = (
    model_all_quotes["days_to_expiry"] / 365.25
)

display(
    model_all_quotes[
        [
            "price_date",
            "option_expiry",
            "days_to_expiry",
            "time_to_expiry",
        ]
    ]
    .drop_duplicates()
    .sort_values("price_date")
    .head()
)

time_counts = (
    model_all_quotes
    .groupby("price_date")["time_to_expiry"]
    .nunique()
)

assert time_counts.eq(1).all()

print(
    "Time-to-expiry range:",
    model_all_quotes["time_to_expiry"].min(),
    "to",
    model_all_quotes["time_to_expiry"].max(),
)

In [ ]:
#calculate log-forward moneyness (= log(strike/forward price))
model_all_quotes["log_moneyness"] = np.log(
    model_all_quotes["strike"]
    / model_all_quotes["forward_price"]
)

display(
    model_all_quotes[
        [
            "price_date",
            "strike",
            "forward_price",
            "log_moneyness",
        ]
    ].head()
)

In [ ]:
#calculate total market implied variance
model_all_quotes["market_total_variance"] = (
    model_all_quotes["iv_decimal"] ** 2
    * model_all_quotes["time_to_expiry"]
)

display(
    model_all_quotes[
        [
            "price_date",
            "strike",
            "option_type",
            "forward_price",
            "iv_decimal",
            "time_to_expiry",
            "log_moneyness",
            "market_total_variance",
        ]
    ].head(10)
)

#variances are actually probably going to have smaller MSE than implied volatilities,
#since implied volatilities < 1

In [ ]:
#validating derived variables
derived_columns = [
    "forward_price",
    "iv_decimal",
    "time_to_expiry",
    "log_moneyness",
    "market_total_variance",
]

derived_values = model_all_quotes[
    derived_columns
].to_numpy()

valid_derived_mask = (
    np.isfinite(derived_values).all(axis=1)
    & (model_all_quotes["forward_price"] > 0)
    & (model_all_quotes["iv_decimal"] > 0)
    & (model_all_quotes["time_to_expiry"] > 0)
    & (
        model_all_quotes["market_total_variance"]
        > 0
    )
)

print(
    "Rows with invalid derived values:",
    (~valid_derived_mask).sum(),
)

assert valid_derived_mask.all()

print("All derived variables are valid.")

In [ ]:
#call and put IVs at the same strike
call_put_comparison = (
    model_all_quotes
    .pivot(
        index=[
            "price_date",
            "option_expiry",
            "strike",
            "forward_price",
            "log_moneyness",
        ],
        columns="option_type",
        values="iv_decimal",
    )
    .reset_index()
)

call_put_comparison.columns.name = None

display(call_put_comparison.head())

complete_call_put_pairs = (
    call_put_comparison
    .dropna(subset=["C", "P"])
    .copy()
)

complete_call_put_pairs[
    "call_minus_put_iv_pp"
] = (
    100
    * (
        complete_call_put_pairs["C"]
        - complete_call_put_pairs["P"]
    )
)

print(
    "Complete call-put pairs:",
    len(complete_call_put_pairs),
)

display(
    complete_call_put_pairs[
        "call_minus_put_iv_pp"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

plt.figure(figsize=(9, 5))

plt.scatter(
    complete_call_put_pairs["log_moneyness"],
    complete_call_put_pairs[
        "call_minus_put_iv_pp"
    ],
    s=10,
    alpha=0.4,
)

plt.axhline(0, linestyle="--")

plt.xlabel("Log-forward moneyness")
plt.ylabel(
    "Call IV minus put IV "
    "(percentage points)"
)
plt.title("Call-Put IV Differences by Moneyness")

plt.tight_layout()
plt.show()

#20,043 pairs where we need to choose to use the call or put
#in theory, call and put IVs should be equal, theseare all negative, so call IV is systematically below the put IV
#from the graph, the differences increase with |k|, better reason not to average
#heuristic: k < 0 means put IV, k >= 0 means call IV

In [ ]:
near_atm_pairs = (
    complete_call_put_pairs
    .assign(
        absolute_log_moneyness=lambda frame:
        frame["log_moneyness"].abs()
    )
    .nsmallest(
        20,
        "absolute_log_moneyness",
    )
)

display(
    near_atm_pairs[
        [
            "price_date",
            "strike",
            "forward_price",
            "log_moneyness",
            "C",
            "P",
            "call_minus_put_iv_pp",
        ]
    ]
)

In [ ]:
#select one OTM per strike for all complete put-call pairs.
is_otm_put = (
    (model_all_quotes["option_type"] == "P")
    & (
        model_all_quotes["strike"]
        < model_all_quotes["forward_price"]
    )
)

is_otm_call = (
    (model_all_quotes["option_type"] == "C")
    & (
        model_all_quotes["strike"]
        >= model_all_quotes["forward_price"]
    )
)

model_data = (
    model_all_quotes.loc[
        is_otm_put | is_otm_call
    ]
    .copy()
)

model_data["quote_selection"] = np.where(
    model_data["option_type"] == "P",
    "OTM put",
    "OTM call",
)

display(
    model_data["quote_selection"]
    .value_counts()
    .to_frame("n_rows")
)

print(
    "Total selected observations:",
    len(model_data),
)

In [ ]:
#verify
selected_key = [
    "price_date",
    "option_expiry",
    "strike",
]

duplicate_selected_mask = (
    model_data.duplicated(
        subset=selected_key,
        keep=False,
    )
)

print(
    "Rows involved in duplicate selected keys:",
    duplicate_selected_mask.sum(),
)

if duplicate_selected_mask.any():
    display(
        model_data.loc[
            duplicate_selected_mask
        ]
        .sort_values(selected_key)
        .head(20)
    )

assert not duplicate_selected_mask.any()

print("At most one OTM quote was selected per strike.")

all_strike_counts = (
    model_all_quotes
    .groupby("price_date")["strike"]
    .nunique()
    .rename("all_valid_strikes")
)

selected_strike_counts = (
    model_data
    .groupby("price_date")["strike"]
    .nunique()
    .rename("selected_otm_strikes")
)

strike_coverage = pd.concat(
    [
        all_strike_counts,
        selected_strike_counts,
    ],
    axis=1,
)

strike_coverage[
    "strikes_without_selected_otm_iv"
] = (
    strike_coverage["all_valid_strikes"]
    - strike_coverage["selected_otm_strikes"]
)

display(strike_coverage.head())

display(
    strike_coverage[
        [
            "all_valid_strikes",
            "selected_otm_strikes",
            "strikes_without_selected_otm_iv",
        ]
    ].describe()
)

#great, lines up mathematically
#now we don't need to force SVI to compromise between the put and call values
#the value in the money is more likely to be affected by its intrinsic price and value 
#instead of just the time value

In [ ]:
#sort
model_data = (
    model_data
    .sort_values(
        [
            "price_date",
            "log_moneyness",
        ]
    )
    .reset_index(drop=True)
)

display(
    model_data[
        [
            "price_date",
            "strike",
            "option_type",
            "quote_selection",
            "forward_price",
            "log_moneyness",
            "iv_decimal",
            "market_total_variance",
        ]
    ].head(10)
)

In [ ]:
#summarize each daily smile
smile_sizes = (
    model_data
    .groupby(
        [
            "price_date",
            "option_expiry",
        ]
    )
    .agg(
        n_options=("strike", "size"),
        n_strikes=("strike", "nunique"),
        n_otm_puts=(
            "option_type",
            lambda values: (values == "P").sum(),
        ),
        n_otm_calls=(
            "option_type",
            lambda values: (values == "C").sum(),
        ),
        minimum_log_moneyness=(
            "log_moneyness",
            "min",
        ),
        maximum_log_moneyness=(
            "log_moneyness",
            "max",
        ),
        forward_price=(
            "forward_price",
            "first",
        ),
        forward_source=(
            "forward_source",
            "first",
        ),
        time_to_expiry=(
            "time_to_expiry",
            "first",
        ),
    )
    .reset_index()
    .sort_values("price_date")
)

display(smile_sizes.head())

display(
    smile_sizes[
        [
            "n_options",
            "n_strikes",
            "n_otm_puts",
            "n_otm_calls",
            "minimum_log_moneyness",
            "maximum_log_moneyness",
            "forward_price",
            "time_to_expiry",
        ]
    ].describe()
)

assert (
    smile_sizes["n_options"]
    == smile_sizes["n_strikes"]
).all()

assert len(smile_sizes) == 103

assert (
    smile_sizes["n_options"] >= 5
).all()

print("Number of daily smiles:", len(smile_sizes))
print("Each selected row represents a distinct strike.")

In [ ]:
#cleaned data summary
final_summary = pd.Series(
    {
        "rows": len(model_data),
        "observation_dates": (
            model_data["price_date"].nunique()
        ),
        "option_expiries": (
            model_data["option_expiry"].nunique()
        ),
        "minimum_strike": (
            model_data["strike"].min()
        ),
        "maximum_strike": (
            model_data["strike"].max()
        ),
        "minimum_forward": (
            model_data["forward_price"].min()
        ),
        "maximum_forward": (
            model_data["forward_price"].max()
        ),
        "minimum_time_to_expiry": (
            model_data["time_to_expiry"].min()
        ),
        "maximum_time_to_expiry": (
            model_data["time_to_expiry"].max()
        ),
        "minimum_log_moneyness": (
            model_data["log_moneyness"].min()
        ),
        "maximum_log_moneyness": (
            model_data["log_moneyness"].max()
        ),
        "minimum_iv": (
            model_data["iv_decimal"].min()
        ),
        "maximum_iv": (
            model_data["iv_decimal"].max()
        ),
        "minimum_total_variance": (
            model_data[
                "market_total_variance"
            ].min()
        ),
        "maximum_total_variance": (
            model_data[
                "market_total_variance"
            ].max()
        ),
    },
    name="value",
)

display(final_summary.to_frame())

In [ ]:
forward_plot_data = (
    model_data[
        [
            "price_date",
            "forward_price",
        ]
    ]
    .drop_duplicates()
    .sort_values("price_date")
)

plt.figure(figsize=(10, 5))

plt.plot(
    forward_plot_data["price_date"],
    forward_plot_data["forward_price"],
)

plt.xlabel("Observation date")
plt.ylabel("ESZ26 futures price")
plt.title("December 2026 Forward Proxy Over Time")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#representative dates
available_dates = (
    model_data["price_date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

representative_dates = [
    available_dates.iloc[0],
    available_dates.iloc[
        len(available_dates) // 2
    ],
    available_dates.iloc[-1],
]

print(
    [
        selected_date.date()
        for selected_date in representative_dates
    ]
)

for selected_date in representative_dates:
    snapshot = model_data.loc[
        model_data["price_date"]
        == selected_date
    ]

    plt.figure(figsize=(9, 5))

    plt.scatter(
        snapshot["log_moneyness"],
        snapshot["iv_decimal"] * 100,
        s=18,
    )

    plt.axvline(
        0,
        linestyle="--",
        alpha=0.7,
    )

    plt.xlabel("Log-forward moneyness")
    plt.ylabel("Market implied volatility (%)")
    plt.title(
        f"Raw OTM IV Smile: "
        f"{selected_date.date()}"
    )

    plt.tight_layout()
    plt.show()
#all their graphs look the same

for selected_date in representative_dates:
    snapshot = model_data.loc[
        model_data["price_date"] == selected_date
    ]

    plt.figure(figsize=(9, 5))

    plt.scatter(
        snapshot["log_moneyness"],
        snapshot["market_total_variance"],
        s=18,
    )

    plt.axvline(
        0,
        linestyle="--",
        alpha=0.7,
    )

    plt.xlabel("Log-forward moneyness")
    plt.ylabel("Market total implied variance")
    plt.title(
        f"Raw Total-Variance Curve: "
        f"{selected_date.date()}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
#overall modeling variables
X = model_data[
    ["log_moneyness"]
].copy()

y = model_data[
    "market_total_variance"
].copy()

groups = model_data[
    "price_date"
].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(
    "Number of observation-date groups:",
    groups.nunique(),
)

In [ ]:
#modeling data for one dat
def get_snapshot_data(
    frame: pd.DataFrame,
    selected_date: pd.Timestamp,
):
    """
    Extract the modeling data for one observation date.

    Returns
    -------
    k_values:
        One-dimensional array of log-moneyness values.

    market_w:
        One-dimensional array of market total variance.

    time_to_expiry:
        The single time-to-expiry value for that date.

    snapshot:
        The corresponding dataframe rows.
    """

    snapshot = (
        frame.loc[
            frame["price_date"] == selected_date
        ]
        .sort_values("log_moneyness")
        .copy()
    )

    if snapshot.empty:
        raise ValueError(
            f"No observations found for "
            f"{selected_date}."
        )

    k_values = snapshot[
        "log_moneyness"
    ].to_numpy()

    market_w = snapshot[
        "market_total_variance"
    ].to_numpy()

    time_values = snapshot[
        "time_to_expiry"
    ].unique()

    if len(time_values) != 1:
        raise ValueError(
            "A fixed-expiry daily curve must "
            "have exactly one value of T."
        )

    time_to_expiry = time_values[0]

    return (
        k_values,
        market_w,
        time_to_expiry,
        snapshot,
    )

first_date = available_dates.iloc[0]

(
    k_first,
    market_w_first,
    T_first,
    first_snapshot,
) = get_snapshot_data(
    model_data,
    first_date,
)

print("First date:", first_date.date())

print(
    "Number of observations:",
    len(k_first),
)

print("k shape:", k_first.shape)

print(
    "Market total-variance shape:",
    market_w_first.shape,
)

print("Time to expiry:", T_first)

print(
    "Log-moneyness range:",
    k_first.min(),
    "to",
    k_first.max(),
)

print(
    "Market total-variance range:",
    market_w_first.min(),
    "to",
    market_w_first.max(),
)

In [ ]:
#final checks
assert len(X) == len(y)
assert len(X) == len(groups)

assert groups.nunique() == 103

assert np.isfinite(
    X["log_moneyness"]
).all()

assert np.isfinite(y).all()

assert (y > 0).all()

assert (
    model_data
    .groupby("price_date")[
        "time_to_expiry"
    ]
    .nunique()
    .eq(1)
    .all()
)

print("Final modeling dataset passed all checks.")

Baseline Model (SVI, independent fitting, no ridge regularization):
Model - SVI; input - log-forward moneyness; output - predicted total implied variance
Selection metric - min. MSE of total implied variance (instate bounds for some parameters)

For each option, record - 
observation date, expiration date, parameter vector, predicted implied volatility, predicted implied total variance, total variance prediction error, implied volatility prediction error

Evaluation - 
Calculate RMSE for total implied variance and implied variability
Calculate parameter scales (SDs) and IQRs, plot their time series

Clarifications - 
SVI is w(k) = a + b[rho(k-m) + sqrt((k-m)^2 + sigma_{SVI}^2)]
parameter bounds are b > 0, -1 < rho < 1, sigma_{SVI} > 0
Fit one SVI curve independently for each observation date (no other date parameters)
Choose the lowest-MSE result among several valid optimization starts (avoid finalizing local minima)

In [ ]:
#svi function
def raw_svi(
    k,
    a,
    b,
    rho,
    m,
    svi_sigma,
):
    """
    Raw SVI total-variance curve.

    Parameters
    ----------
    k:
        Log-forward moneyness.

    a, b, rho, m, svi_sigma:
        Raw SVI parameters.

    Returns
    -------
    Predicted total implied variance.
    """

    k = np.asarray(k, dtype=float)

    return (
        a
        + b
        * (
            rho * (k - m)
            + np.sqrt(
                (k - m) ** 2
                + svi_sigma**2
            )
        )
    )

#a was negative, but that's OK
#hit a bound for m, so widen that range
#now we hit a bound for a, so the plan is to optimize the global minimum then derive a
#that prevents a from affecting the values of the other parameters and guarantees positivity without a bound

Again, every representative date hit at least one boundary.

b = 5 boundary means that the individual raw parameters are not fully identified by the observed section of the curve: the optimizer can increase b and adjust rho, m, and sigma together while producing almost the same fitted curve. So the curve is stable, but its decomposition into the parameters may not be. 

Problem: only 101/103 bound hits, the optimizer did not run correctly, the raw SVI parameters are not being identified freely by the data
b at bound: 81/103 dates. IQR will likely be zero, so we can't use it as a ridge scale either.
Since the observed curve looks nearly linear on the long left side, the data identifies b(1-rho) much better than b and rho individually.

Proposed fix is to add economically meaningful constraints on both SVI asymptotic slopes:
b(1-rho) <= 2 and b(1+rho) <= 2.
Current representative fits have right slopes around 5-10, so the unconstrained extrapolation for outside the observed data is super steep. These constraints can help prevent b from running to 5 just because the right wing has fewer observations and a less strong pattern.

In [ ]:
# Raw SVI parameters that will ultimately be reported
SVI_PARAMETER_NAMES = [
    "a",
    "b",
    "rho",
    "m",
    "svi_sigma",
]

# Positivity-preserving parameters used during optimization
SVI_OPT_PARAMETER_NAMES = [
    "w_min",
    "b",
    "rho",
    "m",
    "svi_sigma",
]

# Maximum permitted left and right asymptotic slopes
WING_SLOPE_LIMIT = 2.0

# A generous upper bound for the minimum variance
W_MIN_UPPER = max(
    2.0,
    2.0
    * model_data[
        "market_total_variance"
    ].max(),
)

SVI_OPT_BOUNDS = [
    (1e-10, W_MIN_UPPER),  # w_min
    (1e-8, 2.0),           # b
    (-0.999, 0.999),       # rho
    (-5.0, 5.0),           # m
    (1e-4, 5.0),           # svi_sigma
]

CONSTRAINT_TOLERANCE = 1e-7

In [ ]:
def optimization_to_raw_theta(phi):
    """
    Convert transformed SVI parameters

        (w_min, b, rho, m, svi_sigma)

    into raw SVI parameters

        (a, b, rho, m, svi_sigma).
    """

    (
        w_min,
        b,
        rho,
        m,
        svi_sigma,
    ) = phi

    square_root_term = np.sqrt(
        max(
            1.0 - rho**2,
            0.0,
        )
    )

    a = (
        w_min
        - b
        * svi_sigma
        * square_root_term
    )

    return np.array(
        [
            a,
            b,
            rho,
            m,
            svi_sigma,
        ],
        dtype=float,
    )

In [ ]:
#curve diagnostics
def svi_global_minimum_from_raw(theta):
    """
    Global minimum total variance of raw SVI.
    """

    a, b, rho, m, svi_sigma = theta

    return (
        a
        + b
        * svi_sigma
        * np.sqrt(
            max(
                1.0 - rho**2,
                0.0,
            )
        )
    )


def svi_minimum_location(theta):
    """
    Log-moneyness at which raw SVI reaches
    its global minimum.
    """

    a, b, rho, m, svi_sigma = theta

    return (
        m
        - rho
        * svi_sigma
        / np.sqrt(
            max(
                1.0 - rho**2,
                1e-12,
            )
        )
    )


def svi_asymptotic_slopes(theta):
    """
    Return the left and right raw-SVI
    asymptotic slopes.
    """

    a, b, rho, m, svi_sigma = theta

    left_slope = b * (1.0 - rho)
    right_slope = b * (1.0 + rho)

    return left_slope, right_slope

In [ ]:
# Weighted total-variance objective

# The lowest-variance 10% of observations receive
# the same capped maximum weight.
WEIGHT_FLOOR_QUANTILE = 0.10


def make_total_variance_weights(
    market_w,
):
    """
    Construct capped inverse-total-variance weights.

    Low-total-variance observations receive more
    weight because a given error in total variance
    corresponds to a larger error in implied
    volatility near the bottom of the curve.

    The variance floor prevents extremely small
    total variances from receiving excessive weight.
    """

    market_w = np.asarray(
        market_w,
        dtype=float,
    )

    variance_floor = max(
        np.quantile(
            market_w,
            WEIGHT_FLOOR_QUANTILE,
        ),
        1e-6,
    )

    weights = (
        1.0
        / np.maximum(
            market_w,
            variance_floor,
        )
    )

    # Normalize weights to have mean 1.
    # This does not change the fitted solution,
    # but keeps the objective on a convenient scale.
    weights = weights / weights.mean()

    return weights, variance_floor


def svi_mse_transformed(
    phi,
    k_values,
    market_w,
):
    """
    Capped inverse-total-variance weighted MSE.

    SVI still predicts total implied variance.
    The weights prevent the high-variance extreme
    left wing from dominating the curve fit.
    """

    raw_theta = optimization_to_raw_theta(
        phi
    )

    predicted_w = raw_svi(
        k_values,
        *raw_theta,
    )

    if not np.isfinite(
        predicted_w
    ).all():
        return 1e12

    errors = predicted_w - market_w

    weights, _ = (
        make_total_variance_weights(
            market_w
        )
    )

    weighted_mse = np.average(
        errors**2,
        weights=weights,
    )

    return weighted_mse

In [ ]:
#12 starting points
def make_svi_initial_guesses(
    k_values,
    market_w,
):
    """
    Create 12 deterministic and feasible
    initial guesses for one daily SVI fit.
    """

    k_values = np.asarray(
        k_values,
        dtype=float,
    )

    market_w = np.asarray(
        market_w,
        dtype=float,
    )

    observed_minimum_w = market_w.min()

    w_min_start = np.clip(
        0.80 * observed_minimum_w,
        SVI_OPT_BOUNDS[0][0] + 1e-8,
        SVI_OPT_BOUNDS[0][1] - 1e-8,
    )

    central_k_range = max(
        np.quantile(k_values, 0.95)
        - np.quantile(k_values, 0.05),
        1e-4,
    )

    central_w_range = max(
        np.quantile(market_w, 0.95)
        - np.quantile(market_w, 0.05),
        1e-4,
    )

    b_base = np.clip(
        central_w_range
        / central_k_range,
        0.005,
        1.0,
    )

    m_at_market_minimum = k_values[
        np.argmin(market_w)
    ]

    m_candidates = [
        m_at_market_minimum,
        0.0,
    ]

    # rho, sigma, b multiplier
    start_specs = [
        (-0.80, 0.10, 0.50),
        (-0.40, 0.30, 1.00),
        (0.00, 0.80, 1.00),
        (0.40, 0.30, 1.00),
        (0.80, 0.80, 1.00),
        (0.90, 1.50, 0.50),
    ]

    starts = []

    for m_start in m_candidates:
        for (
            rho_start,
            sigma_start,
            b_multiplier,
        ) in start_specs:

            requested_b = (
                b_base * b_multiplier
            )

            # Largest b that satisfies both
            # wing-slope constraints.
            feasible_b_upper = min(
                SVI_OPT_BOUNDS[1][1],
                WING_SLOPE_LIMIT
                / (1.0 - rho_start),
                WING_SLOPE_LIMIT
                / (1.0 + rho_start),
            )

            b_start = np.clip(
                requested_b,
                SVI_OPT_BOUNDS[1][0]
                + 1e-8,
                0.95 * feasible_b_upper,
            )

            start = np.array(
                [
                    w_min_start,
                    b_start,
                    rho_start,
                    m_start,
                    sigma_start,
                ],
                dtype=float,
            )

            for parameter_index, (
                lower,
                upper,
            ) in enumerate(
                SVI_OPT_BOUNDS
            ):
                start[
                    parameter_index
                ] = np.clip(
                    start[
                        parameter_index
                    ],
                    lower + 1e-8,
                    upper - 1e-8,
                )

            starts.append(start)

    return starts

test_starts = make_svi_initial_guesses(
    k_first,
    market_w_first,
)

print(
    "Number of initial guesses:",
    len(test_starts),
)

assert len(test_starts) == 12

In [ ]:
#boundary and constraint diagnostics
def identify_box_bound_hits(
    values,
    names,
    bounds,
    tolerance=1e-5,
):
    """
    Identify optimization parameters that
    finished near an ordinary box bound.
    """

    hits = []

    for (
        value,
        name,
        (lower, upper),
    ) in zip(
        values,
        names,
        bounds,
    ):
        near_lower = np.isclose(
            value,
            lower,
            atol=tolerance,
            rtol=0,
        )

        near_upper = np.isclose(
            value,
            upper,
            atol=tolerance,
            rtol=0,
        )

        if near_lower or near_upper:
            hits.append(name)

    return hits

def identify_active_slope_constraints(
    left_slope,
    right_slope,
    tolerance=1e-5,
):
    """
    Identify wing-slope constraints that
    are active at the fitted solution.
    """

    active_constraints = []

    if np.isclose(
        left_slope,
        WING_SLOPE_LIMIT,
        atol=tolerance,
        rtol=0,
    ):
        active_constraints.append(
            "left_slope_limit"
        )

    if np.isclose(
        right_slope,
        WING_SLOPE_LIMIT,
        atol=tolerance,
        rtol=0,
    ):
        active_constraints.append(
            "right_slope_limit"
        )

    return active_constraints

In [ ]:
# Fit one daily SVI curve using the
# weighted total-variance objective

def fit_svi_snapshot(
    k_values,
    market_w,
):
    """
    Fit constrained raw SVI independently
    to one date.

    The fitting objective is capped,
    inverse-total-variance weighted MSE.

    Optimization parameters:
        (w_min, b, rho, m, svi_sigma)

    Reported raw parameters:
        (a, b, rho, m, svi_sigma)
    """

    k_values = np.asarray(
        k_values,
        dtype=float,
    )

    market_w = np.asarray(
        market_w,
        dtype=float,
    )

    initial_guesses = (
        make_svi_initial_guesses(
            k_values,
            market_w,
        )
    )

    # Calculate the date-specific weights once.
    weights, variance_floor = (
        make_total_variance_weights(
            market_w
        )
    )

    wing_slope_constraints = [
        {
            "type": "ineq",
            "fun": lambda phi: (
                WING_SLOPE_LIMIT
                - phi[1]
                * (1.0 - phi[2])
            ),
        },
        {
            "type": "ineq",
            "fun": lambda phi: (
                WING_SLOPE_LIMIT
                - phi[1]
                * (1.0 + phi[2])
            ),
        },
    ]

    candidates = []

    for start_index, initial_guess in enumerate(
        initial_guesses
    ):
        result = minimize(
            fun=svi_mse_transformed,
            x0=initial_guess,
            args=(
                k_values,
                market_w,
            ),
            method="SLSQP",
            bounds=SVI_OPT_BOUNDS,
            constraints=wing_slope_constraints,
            options={
                "maxiter": 3000,
                "ftol": 1e-12,
                "disp": False,
            },
        )

        phi = np.asarray(
            result.x,
            dtype=float,
        )

        raw_theta = (
            optimization_to_raw_theta(
                phi
            )
        )

        predicted_w = raw_svi(
            k_values,
            *raw_theta,
        )

        (
            left_slope,
            right_slope,
        ) = svi_asymptotic_slopes(
            raw_theta
        )

        valid = (
            np.isfinite(phi).all()
            and np.isfinite(
                raw_theta
            ).all()
            and np.isfinite(
                predicted_w
            ).all()
            and np.all(
                predicted_w > 0
            )
            and phi[0] > 0
            and (
                left_slope
                <= WING_SLOPE_LIMIT
                + CONSTRAINT_TOLERANCE
            )
            and (
                right_slope
                <= WING_SLOPE_LIMIT
                + CONSTRAINT_TOLERANCE
            )
        )

        if not valid:
            continue

        errors = (
            predicted_w
            - market_w
        )

        # This is the objective used to choose
        # among the 12 candidate solutions.
        weighted_mse = np.average(
            errors**2,
            weights=weights,
        )

        # Retain the ordinary MSE as a diagnostic.
        unweighted_mse = np.mean(
            errors**2
        )

        candidates.append(
            {
                "result": result,
                "phi": phi,
                "theta": raw_theta,
                "weighted_mse": (
                    weighted_mse
                ),
                "unweighted_mse": (
                    unweighted_mse
                ),
                "start_index": (
                    start_index
                ),
            }
        )

    if not candidates:
        raise RuntimeError(
            "No valid constrained weighted "
            "SVI fit was found."
        )

    successful_candidates = [
        candidate
        for candidate in candidates
        if candidate[
            "result"
        ].success
    ]

    candidate_pool = (
        successful_candidates
        if successful_candidates
        else candidates
    )

    # Select the best candidate using the
    # weighted objective, not ordinary MSE.
    best_candidate = min(
        candidate_pool,
        key=lambda candidate:
        candidate["weighted_mse"],
    )

    best_result = best_candidate[
        "result"
    ]

    best_phi = best_candidate[
        "phi"
    ]

    best_theta = best_candidate[
        "theta"
    ]

    (
        left_slope,
        right_slope,
    ) = svi_asymptotic_slopes(
        best_theta
    )

    box_bound_hits = (
        identify_box_bound_hits(
            best_phi,
            SVI_OPT_PARAMETER_NAMES,
            SVI_OPT_BOUNDS,
        )
    )

    active_slope_constraints = (
        identify_active_slope_constraints(
            left_slope,
            right_slope,
        )
    )

    return {
        "theta": best_theta,
        "optimization_theta": best_phi,

        # Keep "mse" so the existing preflight
        # code continues to work.
        "mse": best_candidate[
            "weighted_mse"
        ],

        "weighted_mse": best_candidate[
            "weighted_mse"
        ],

        "unweighted_mse": best_candidate[
            "unweighted_mse"
        ],

        "weight_variance_floor": (
            variance_floor
        ),

        "optimizer_success": bool(
            best_result.success
        ),

        "optimizer_message": str(
            best_result.message
        ),

        "n_iterations": (
            best_result.nit
        ),

        "n_function_evaluations": (
            best_result.nfev
        ),

        "best_start_index": (
            best_candidate[
                "start_index"
            ]
        ),

        "n_initial_starts": (
            len(initial_guesses)
        ),

        "n_valid_starts": (
            len(candidates)
        ),

        "n_successful_starts": (
            len(
                successful_candidates
            )
        ),

        "global_minimum_w": (
            best_phi[0]
        ),

        "minimum_location_k": (
            svi_minimum_location(
                best_theta
            )
        ),

        "left_asymptotic_slope": (
            left_slope
        ),

        "right_asymptotic_slope": (
            right_slope
        ),

        "box_bound_hit_names": (
            box_bound_hits
        ),

        "active_constraint_names": (
            active_slope_constraints
        ),

        "n_box_bound_hits": (
            len(box_bound_hits)
        ),

        "n_active_constraints": (
            len(
                active_slope_constraints
            )
        ),
    }

In [ ]:
#test the first date
first_fit = fit_svi_snapshot(
    k_first,
    market_w_first,
)

first_theta = first_fit["theta"]

first_phi = first_fit[
    "optimization_theta"
]

print("Raw SVI parameters:")

print(
    dict(
        zip(
            SVI_PARAMETER_NAMES,
            first_theta,
        )
    )
)

print("\nOptimization parameters:")

print(
    dict(
        zip(
            SVI_OPT_PARAMETER_NAMES,
            first_phi,
        )
    )
)

print(
    "\nTotal-variance MSE:",
    first_fit["mse"],
)

print(
    "Optimizer success:",
    first_fit["optimizer_success"],
)

print(
    "Optimizer message:",
    first_fit["optimizer_message"],
)

print(
    "Global minimum fitted variance:",
    first_fit[
        "global_minimum_w"
    ],
)

print(
    "Minimum location k:",
    first_fit[
        "minimum_location_k"
    ],
)

print(
    "Left asymptotic slope:",
    first_fit[
        "left_asymptotic_slope"
    ],
)

print(
    "Right asymptotic slope:",
    first_fit[
        "right_asymptotic_slope"
    ],
)

print(
    "Box-bound hits:",
    first_fit[
        "box_bound_hit_names"
    ],
)

print(
    "Active slope constraints:",
    first_fit[
        "active_constraint_names"
    ],
)

print(
    "Successful starts:",
    first_fit[
        "n_successful_starts"
    ],
    "/",
    first_fit[
        "n_initial_starts"
    ],
)

This fit is numerically stable and economically plausible. No arbitrary box-bound hits, all starting points converged successfully, total variance is positive, reasonable fitting minimum at k = 0.385, left slope is the same, only the right wing bound was hit.
MSE is only 3.4% worse, which is not that bad.

After the changes, this looks way better numerically! Cannot directly compare weighted MSE with previous total MSE though.

In [ ]:
print(
    "Weighted total-variance MSE:",
    first_fit["weighted_mse"],
)

print(
    "Unweighted total-variance MSE:",
    first_fit["unweighted_mse"],
)

In [ ]:
#plots
first_predicted_w = raw_svi(
    k_first,
    *first_theta,
)

first_predicted_iv = np.sqrt(
    first_predicted_w / T_first
)

first_plot_order = np.argsort(k_first)

plt.figure(figsize=(9, 5))

plt.scatter(
    k_first,
    market_w_first,
    s=18,
    label="Market",
)

plt.plot(
    k_first[first_plot_order],
    first_predicted_w[first_plot_order],
    label="Constrained SVI",
)

plt.axvline(
    0,
    linestyle="--",
    alpha=0.7,
    label="ATM",
)

plt.axvline(
    first_fit["minimum_location_k"],
    linestyle=":",
    alpha=0.7,
    label="Fitted minimum",
)

plt.xlabel("Log-forward moneyness")
plt.ylabel("Total implied variance")

plt.title(
    f"First Constrained SVI Fit: "
    f"{first_date.date()}"
)

plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))

plt.scatter(
    k_first,
    first_snapshot["iv_decimal"] * 100,
    s=18,
    label="Market",
)

plt.plot(
    k_first[first_plot_order],
    first_predicted_iv[first_plot_order] * 100,
    label="Constrained SVI",
)

plt.axvline(
    0,
    linestyle="--",
    alpha=0.7,
    label="ATM",
)

plt.axvline(
    first_fit["minimum_location_k"],
    linestyle=":",
    alpha=0.7,
    label="Fitted minimum",
)

plt.xlabel("Log-forward moneyness")
plt.ylabel("Implied volatility (%)")

plt.title(
    f"First Constrained SVI Fit in IV: "
    f"{first_date.date()}"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#representative data preflight
# Representative-date preflight

preflight_records = []
preflight_fits = {}

for selected_date in representative_dates:
    (
        k_test,
        w_test,
        T_test,
        snapshot_test,
    ) = get_snapshot_data(
        model_data,
        selected_date,
    )

    test_fit = fit_svi_snapshot(
        k_test,
        w_test,
    )

    preflight_fits[selected_date] = {
        "fit": test_fit,
        "k": k_test,
        "market_w": w_test,
        "T": T_test,
        "snapshot": snapshot_test,
    }

    preflight_records.append(
        {
            "price_date": selected_date,

            "weighted_mse": test_fit[
                "weighted_mse"
            ],

            "unweighted_mse": test_fit[
                "unweighted_mse"
            ],

            "optimizer_success": test_fit[
                "optimizer_success"
            ],

            "w_min": test_fit[
                "global_minimum_w"
            ],

            "minimum_location_k": test_fit[
                "minimum_location_k"
            ],

            "left_slope": test_fit[
                "left_asymptotic_slope"
            ],

            "right_slope": test_fit[
                "right_asymptotic_slope"
            ],

            "successful_starts": test_fit[
                "n_successful_starts"
            ],

            "valid_starts": test_fit[
                "n_valid_starts"
            ],

            "box_bound_hits": ", ".join(
                test_fit[
                    "box_bound_hit_names"
                ]
            ),

            "active_constraints": ", ".join(
                test_fit[
                    "active_constraint_names"
                ]
            ),
        }
    )

preflight_results = pd.DataFrame(
    preflight_records
)

display(preflight_results)

#validate
assert preflight_results[
    "optimizer_success"
].all()

assert (
    preflight_results["w_min"] > 0
).all()

assert (
    preflight_results["left_slope"]
    <= WING_SLOPE_LIMIT
    + CONSTRAINT_TOLERANCE
).all()

assert (
    preflight_results["right_slope"]
    <= WING_SLOPE_LIMIT
    + CONSTRAINT_TOLERANCE
).all()

print(
    "All representative-date fits "
    "passed the numerical checks."
)

#plot
for selected_date in representative_dates:
    stored_result = preflight_fits[
        selected_date
    ]

    test_fit = stored_result["fit"]
    k_test = stored_result["k"]
    w_test = stored_result["market_w"]
    T_test = stored_result["T"]
    snapshot_test = stored_result[
        "snapshot"
    ]

    theta_test = test_fit["theta"]

    predicted_w_test = raw_svi(
        k_test,
        *theta_test,
    )

    predicted_iv_test = np.sqrt(
        predicted_w_test / T_test
    )

    plot_order = np.argsort(k_test)

    plt.figure(figsize=(9, 5))

    plt.scatter(
        k_test,
        w_test,
        s=18,
        label="Market",
    )

    plt.plot(
        k_test[plot_order],
        predicted_w_test[plot_order],
        label="Constrained SVI",
    )

    plt.axvline(
        0,
        linestyle="--",
        alpha=0.7,
        label="ATM",
    )

    plt.axvline(
        test_fit["minimum_location_k"],
        linestyle=":",
        alpha=0.7,
        label="Fitted minimum",
    )

    plt.xlabel("Log-forward moneyness")
    plt.ylabel("Total implied variance")

    plt.title(
        f"Preflight Total-Variance Fit: "
        f"{selected_date.date()}"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))

    plt.scatter(
        k_test,
        snapshot_test["iv_decimal"] * 100,
        s=18,
        label="Market",
    )

    plt.plot(
        k_test[plot_order],
        predicted_iv_test[plot_order] * 100,
        label="Constrained SVI",
    )

    plt.axvline(
        0,
        linestyle="--",
        alpha=0.7,
        label="ATM",
    )

    plt.axvline(
        test_fit["minimum_location_k"],
        linestyle=":",
        alpha=0.7,
        label="Fitted minimum",
    )

    plt.xlabel("Log-forward moneyness")
    plt.ylabel("Implied volatility (%)")

    plt.title(
        f"Preflight IV Fit: "
        f"{selected_date.date()}"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

Only warning is that for 6/1, the curve is trying to place the global minimum at zero (w_min bound hit).

Problem is that here data is a check mark but fit is giving monotonically decreasing.
Potential next fix: IV-equivalent weighted total-variance loss.

The weighted-loss version resolves the main numerical issue.The representative fits now have positive w_min, minimum locations around k = 0.20-0.29, no box-bound hits, all 12 starts successful, and very stable estimated left slopes.

The unweighted MSEs increased, but that's expected.

In [ ]:
june_date = pd.Timestamp(
    "2026-06-01"
)

(
    k_june,
    market_w_june,
    T_june,
    june_snapshot,
) = get_snapshot_data(
    model_data,
    june_date,
)

june_fit = fit_svi_snapshot(
    k_june,
    market_w_june,
)

june_predicted_w = raw_svi(
    k_june,
    *june_fit["theta"],
)

june_predicted_iv = np.sqrt(
    june_predicted_w / T_june
)

june_diagnostics = (
    june_snapshot.copy()
)

june_diagnostics[
    "predicted_w"
] = june_predicted_w

june_diagnostics[
    "w_error"
] = (
    june_predicted_w
    - market_w_june
)

june_diagnostics[
    "squared_w_error"
] = (
    june_diagnostics[
        "w_error"
    ] ** 2
)

june_diagnostics[
    "predicted_iv"
] = june_predicted_iv

june_diagnostics[
    "iv_error_pp"
] = (
    100
    * (
        june_predicted_iv
        - june_diagnostics[
            "iv_decimal"
        ]
    )
)

june_diagnostics[
    "region"
] = pd.cut(
    june_diagnostics[
        "log_moneyness"
    ],
    bins=[
        -np.inf,
        -2.0,
        -0.5,
        0.2,
        np.inf,
    ],
    labels=[
        "extreme_left",
        "left",
        "central",
        "right",
    ],
)

total_squared_w_error = (
    june_diagnostics[
        "squared_w_error"
    ].sum()
)

june_region_summary = (
    june_diagnostics
    .groupby(
        "region",
        observed=True,
    )
    .agg(
        n_quotes=(
            "strike",
            "size",
        ),
        market_w_min=(
            "market_total_variance",
            "min",
        ),
        market_w_max=(
            "market_total_variance",
            "max",
        ),
        mean_squared_w_error=(
            "squared_w_error",
            "mean",
        ),
        total_squared_w_error=(
            "squared_w_error",
            "sum",
        ),
        rmse_iv_pp=(
            "iv_error_pp",
            lambda errors:
            np.sqrt(
                np.mean(errors**2)
            ),
        ),
    )
    .reset_index()
)

june_region_summary[
    "share_of_total_w_loss"
] = (
    june_region_summary[
        "total_squared_w_error"
    ]
    / total_squared_w_error
)

display(june_region_summary)

#conomically important area
june_zoom = (
    june_diagnostics[
        june_diagnostics[
            "log_moneyness"
        ] >= -0.5
    ]
    .sort_values(
        "log_moneyness"
    )
)

plt.figure(figsize=(9, 5))

plt.scatter(
    june_zoom[
        "log_moneyness"
    ],
    june_zoom[
        "iv_decimal"
    ] * 100,
    s=22,
    label="Market",
)

plt.plot(
    june_zoom[
        "log_moneyness"
    ],
    june_zoom[
        "predicted_iv"
    ] * 100,
    label="Current constrained SVI",
)

plt.axvline(
    june_fit[
        "minimum_location_k"
    ],
    linestyle=":",
    label="Fitted minimum",
)

plt.xlabel("Log-forward moneyness")
plt.ylabel("Implied volatility (%)")

plt.title(
    "June 1 Fit — Central and Right Wing"
)

plt.legend()
plt.tight_layout()
plt.show()

Right now, five extreme-left quotes account for 58.9% of the total-variance loss.
Elevent right-wing quotes account for only 12.6%, even though their IV RMSE is 10.43 percentage points.
So basically it's saying that fitting five tail observations with very high variance is more important than correctly capturing the entire trough and upturn, which is not true.
Replacing the objective by weighting the squared errors approximately according to how they translate into IV errors.

(As mentioned above, this is now fixed with weighted MSE selection metric.)

In [ ]:
# Fit the weighted, constrained SVI model
# independently for every observation date.

parameter_records = []
prediction_frames = []

for date_index, selected_date in enumerate(
    available_dates,
    start=1,
):
    (
        k_values,
        market_w,
        time_to_expiry,
        snapshot,
    ) = get_snapshot_data(
        model_data,
        selected_date,
    )

    # Fit one SVI curve for this date
    fit = fit_svi_snapshot(
        k_values,
        market_w,
    )

    theta = fit["theta"]

    # Generate fitted total variance and IV
    predicted_w = raw_svi(
        k_values,
        *theta,
    )

    predicted_iv = np.sqrt(
        predicted_w / time_to_expiry
    )

    # Recreate the observation weights so they
    # can be stored with each prediction
    weights, variance_floor = (
        make_total_variance_weights(
            market_w
        )
    )

    current_predictions = snapshot.copy()

    # Predictions and total-variance errors
    current_predictions[
        "predicted_total_variance"
    ] = predicted_w

    current_predictions[
        "total_variance_error"
    ] = (
        predicted_w - market_w
    )

    current_predictions[
        "squared_total_variance_error"
    ] = (
        current_predictions[
            "total_variance_error"
        ] ** 2
    )

    # Weighted objective components
    current_predictions[
        "total_variance_weight"
    ] = weights

    current_predictions[
        "weighted_squared_total_variance_error"
    ] = (
        weights
        * current_predictions[
            "squared_total_variance_error"
        ]
    )

    current_predictions[
        "weight_variance_floor"
    ] = variance_floor

    # Implied-volatility predictions and errors
    current_predictions[
        "predicted_iv"
    ] = predicted_iv

    current_predictions[
        "iv_error"
    ] = (
        predicted_iv
        - current_predictions[
            "iv_decimal"
        ]
    )

    current_predictions[
        "iv_error_pp"
    ] = (
        100.0
        * current_predictions[
            "iv_error"
        ]
    )

    current_predictions[
        "squared_iv_error"
    ] = (
        current_predictions[
            "iv_error"
        ] ** 2
    )

    # Attach the fitted raw SVI parameters
    # to every observation from this date
    for parameter_name, value in zip(
        SVI_PARAMETER_NAMES,
        theta,
    ):
        current_predictions[
            parameter_name
        ] = value

    # Attach derived curve quantities
    current_predictions[
        "w_min"
    ] = fit["global_minimum_w"]

    current_predictions[
        "minimum_location_k"
    ] = fit["minimum_location_k"]

    current_predictions[
        "left_asymptotic_slope"
    ] = fit["left_asymptotic_slope"]

    current_predictions[
        "right_asymptotic_slope"
    ] = fit["right_asymptotic_slope"]

    prediction_frames.append(
        current_predictions
    )

    # Save one parameter and diagnostic record
    # for this date
    parameter_records.append(
        {
            "price_date": selected_date,

            "option_expiry": snapshot[
                "option_expiry"
            ].iloc[0],

            "n_options": len(snapshot),

            # Raw SVI parameters
            "a": theta[0],
            "b": theta[1],
            "rho": theta[2],
            "m": theta[3],
            "svi_sigma": theta[4],

            # Derived curve quantities
            "w_min": fit[
                "global_minimum_w"
            ],

            "minimum_location_k": fit[
                "minimum_location_k"
            ],

            "left_asymptotic_slope": fit[
                "left_asymptotic_slope"
            ],

            "right_asymptotic_slope": fit[
                "right_asymptotic_slope"
            ],

            # Fitting objectives
            "objective_weighted_mse": fit[
                "weighted_mse"
            ],

            "objective_unweighted_mse": fit[
                "unweighted_mse"
            ],

            "weight_variance_floor": fit[
                "weight_variance_floor"
            ],

            # Optimizer diagnostics
            "optimizer_success": fit[
                "optimizer_success"
            ],

            "optimizer_message": fit[
                "optimizer_message"
            ],

            "n_iterations": fit[
                "n_iterations"
            ],

            "n_function_evaluations": fit[
                "n_function_evaluations"
            ],

            # Multiple-start diagnostics
            "best_start_index": fit[
                "best_start_index"
            ],

            "n_initial_starts": fit[
                "n_initial_starts"
            ],

            "n_valid_starts": fit[
                "n_valid_starts"
            ],

            "n_successful_starts": fit[
                "n_successful_starts"
            ],

            # Ordinary parameter-bound diagnostics
            "n_box_bound_hits": fit[
                "n_box_bound_hits"
            ],

            "box_bound_hit_names": ", ".join(
                fit[
                    "box_bound_hit_names"
                ]
            ),

            # Active slope constraints
            "n_active_constraints": fit[
                "n_active_constraints"
            ],

            "active_constraint_names": ", ".join(
                fit[
                    "active_constraint_names"
                ]
            ),
        }
    )

    # Progress update
    if (
        date_index % 10 == 0
        or date_index
        == len(available_dates)
    ):
        print(
            f"Completed "
            f"{date_index} / "
            f"{len(available_dates)} dates"
        )

In [ ]:
baseline_predictions = (
    pd.concat(
        prediction_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "price_date",
            "log_moneyness",
        ]
    )
    .reset_index(drop=True)
)

baseline_parameters = (
    pd.DataFrame(
        parameter_records
    )
    .sort_values(
        "price_date"
    )
    .reset_index(drop=True)
)

print(
    "Prediction rows:",
    len(baseline_predictions),
)

print(
    "Parameter rows:",
    len(baseline_parameters),
)

display(
    baseline_parameters.head()
)

In [ ]:
#validate all fits
assert (
    len(baseline_predictions)
    == len(model_data)
)

assert (
    len(baseline_parameters)
    == len(available_dates)
)

assert (
    baseline_predictions[
        "price_date"
    ].nunique()
    == len(available_dates)
)

assert baseline_predictions[
    "predicted_total_variance"
].notna().all()

assert (
    baseline_predictions[
        "predicted_total_variance"
    ] > 0
).all()

assert baseline_predictions[
    "predicted_iv"
].notna().all()

assert (
    baseline_predictions[
        "predicted_iv"
    ] > 0
).all()

assert baseline_parameters[
    "optimizer_success"
].all()

assert (
    baseline_parameters[
        "w_min"
    ] > 0
).all()

assert (
    baseline_parameters[
        "left_asymptotic_slope"
    ]
    <= WING_SLOPE_LIMIT
    + CONSTRAINT_TOLERANCE
).all()

assert (
    baseline_parameters[
        "right_asymptotic_slope"
    ]
    <= WING_SLOPE_LIMIT
    + CONSTRAINT_TOLERANCE
).all()

assert (
    baseline_parameters[
        "n_valid_starts"
    ] > 0
).all()

print(
    "All 103 weighted constrained "
    "SVI fits passed validation."
)

In [ ]:
#objectives match predictions?
recomputed_daily_objectives = (
    baseline_predictions
    .groupby(
        [
            "price_date",
            "option_expiry",
        ]
    )
    .agg(
        recomputed_weighted_mse=(
            "weighted_squared_total_variance_error",
            "mean",
        ),
        recomputed_unweighted_mse=(
            "squared_total_variance_error",
            "mean",
        ),
        mean_weight=(
            "total_variance_weight",
            "mean",
        ),
    )
    .reset_index()
)

objective_check = (
    baseline_parameters
    .merge(
        recomputed_daily_objectives,
        on=[
            "price_date",
            "option_expiry",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert np.allclose(
    objective_check[
        "objective_weighted_mse"
    ],
    objective_check[
        "recomputed_weighted_mse"
    ],
    rtol=1e-7,
    atol=1e-12,
)

assert np.allclose(
    objective_check[
        "objective_unweighted_mse"
    ],
    objective_check[
        "recomputed_unweighted_mse"
    ],
    rtol=1e-7,
    atol=1e-12,
)

assert np.allclose(
    objective_check[
        "mean_weight"
    ],
    1.0,
    rtol=1e-10,
    atol=1e-10,
)

print(
    "Stored and recomputed "
    "objective values agree."
)

In [ ]:
#optimizer and constraint diagnostics
print(
    "Successful optimizer fits:",
    baseline_parameters[
        "optimizer_success"
    ].sum(),
    "/",
    len(baseline_parameters),
)

print(
    "Dates with all starts successful:",
    (
        baseline_parameters[
            "n_successful_starts"
        ]
        == baseline_parameters[
            "n_initial_starts"
        ]
    ).sum(),
    "/",
    len(baseline_parameters),
)

print(
    "Dates with at least one "
    "box-bound hit:",
    (
        baseline_parameters[
            "n_box_bound_hits"
        ] > 0
    ).sum(),
    "/",
    len(baseline_parameters),
)

print(
    "Dates with at least one "
    "active slope constraint:",
    (
        baseline_parameters[
            "n_active_constraints"
        ] > 0
    ).sum(),
    "/",
    len(baseline_parameters),
)

box_bound_summary = (
    baseline_parameters[
        "box_bound_hit_names"
    ]
    .fillna("")
    .replace(
        "",
        "none",
    )
    .value_counts()
    .rename_axis(
        "box_bound_combination"
    )
    .to_frame(
        "n_dates"
    )
)

display(
    box_bound_summary
)

active_constraint_summary = (
    baseline_parameters[
        "active_constraint_names"
    ]
    .fillna("")
    .replace(
        "",
        "none",
    )
    .value_counts()
    .rename_axis(
        "active_constraint_combination"
    )
    .to_frame(
        "n_dates"
    )
)

display(
    active_constraint_summary
)

individual_box_bound_records = []

for parameter in SVI_OPT_PARAMETER_NAMES:
    hit_mask = (
        baseline_parameters[
            "box_bound_hit_names"
        ]
        .fillna("")
        .str.split(", ")
        .apply(
            lambda names:
            parameter in names
        )
    )

    individual_box_bound_records.append(
        {
            "parameter": parameter,
            "n_dates_at_bound": int(
                hit_mask.sum()
            ),
            "percent_of_dates": (
                100.0
                * hit_mask.mean()
            ),
        }
    )

individual_box_bound_summary = (
    pd.DataFrame(
        individual_box_bound_records
    )
)

display(
    individual_box_bound_summary
)

Good! No dates forced any of the parameters to an artifical limit.
97/103 right wing controls means that observed data usually prefers an eventual right-wing slope above two.

In [ ]:
# Add each date's observed log-moneyness range.
# This version is safe to rerun.

columns_to_remove = [
    "observed_k_min",
    "observed_k_max",
    "observed_k_min_x",
    "observed_k_min_y",
    "observed_k_max_x",
    "observed_k_max_y",
    "minimum_inside_observed_range",
]

baseline_parameters = (
    baseline_parameters
    .drop(
        columns=columns_to_remove,
        errors="ignore",
    )
)

daily_moneyness_ranges = (
    model_data
    .groupby(
        "price_date",
        as_index=False,
    )
    .agg(
        observed_k_min=(
            "log_moneyness",
            "min",
        ),
        observed_k_max=(
            "log_moneyness",
            "max",
        ),
    )
)

baseline_parameters = (
    baseline_parameters
    .merge(
        daily_moneyness_ranges,
        on="price_date",
        how="left",
        validate="one_to_one",
    )
)

assert baseline_parameters[
    "observed_k_min"
].notna().all()

assert baseline_parameters[
    "observed_k_max"
].notna().all()

baseline_parameters[
    "minimum_inside_observed_range"
] = (
    baseline_parameters[
        "minimum_location_k"
    ]
    .between(
        baseline_parameters[
            "observed_k_min"
        ],
        baseline_parameters[
            "observed_k_max"
        ],
    )
)

print(
    "Fitted minima inside the "
    "observed moneyness range:",
    baseline_parameters[
        "minimum_inside_observed_range"
    ].sum(),
    "/",
    len(baseline_parameters),
)

baseline_parameters[
    "minimum_inside_observed_range"
] = (
    baseline_parameters[
        "minimum_location_k"
    ]
    .between(
        baseline_parameters[
            "observed_k_min"
        ],
        baseline_parameters[
            "observed_k_max"
        ],
    )
)

print(
    "Fitted minima inside the "
    "observed moneyness range:",
    baseline_parameters[
        "minimum_inside_observed_range"
    ].sum(),
    "/",
    len(baseline_parameters),
)

display(
    baseline_parameters[
        [
            "price_date",
            "w_min",
            "minimum_location_k",
            "observed_k_min",
            "observed_k_max",
            "minimum_inside_observed_range",
            "left_asymptotic_slope",
            "right_asymptotic_slope",
            "box_bound_hit_names",
            "active_constraint_names",
        ]
    ].head(10)
)

In [ ]:
#overall fitting metrics
overall_weighted_mse_total_variance = (
    baseline_predictions[
        "weighted_squared_total_variance_error"
    ].mean()
)

overall_unweighted_mse_total_variance = (
    baseline_predictions[
        "squared_total_variance_error"
    ].mean()
)

overall_rmse_total_variance = np.sqrt(
    overall_unweighted_mse_total_variance
)

overall_rmse_iv_decimal = np.sqrt(
    baseline_predictions[
        "squared_iv_error"
    ].mean()
)

overall_rmse_iv_pp = (
    100.0
    * overall_rmse_iv_decimal
)

overall_metrics = pd.Series(
    {
        "n_observations": (
            len(
                baseline_predictions
            )
        ),
        "n_dates": (
            baseline_predictions[
                "price_date"
            ].nunique()
        ),
        "weighted_mse_total_variance": (
            overall_weighted_mse_total_variance
        ),
        "unweighted_mse_total_variance": (
            overall_unweighted_mse_total_variance
        ),
        "unweighted_rmse_total_variance": (
            overall_rmse_total_variance
        ),
        "rmse_iv_decimal": (
            overall_rmse_iv_decimal
        ),
        "rmse_iv_percentage_points": (
            overall_rmse_iv_pp
        ),
    },
    name="value",
)

display(
    overall_metrics.to_frame()
)

#daily fitting metrics
baseline_daily_metrics = (
    baseline_predictions
    .groupby(
        [
            "price_date",
            "option_expiry",
        ]
    )
    .agg(
        n_quotes=(
            "strike",
            "size",
        ),

        weighted_mse_total_variance=(
            "weighted_squared_total_variance_error",
            "mean",
        ),

        unweighted_mse_total_variance=(
            "squared_total_variance_error",
            "mean",
        ),

        rmse_total_variance=(
            "total_variance_error",
            lambda errors:
            np.sqrt(
                np.mean(
                    errors**2
                )
            ),
        ),

        rmse_iv_decimal=(
            "iv_error",
            lambda errors:
            np.sqrt(
                np.mean(
                    errors**2
                )
            ),
        ),

        rmse_iv_percentage_points=(
            "iv_error_pp",
            lambda errors:
            np.sqrt(
                np.mean(
                    errors**2
                )
            ),
        ),

        mean_iv_error_pp=(
            "iv_error_pp",
            "mean",
        ),

        maximum_absolute_iv_error_pp=(
            "iv_error_pp",
            lambda errors:
            np.max(
                np.abs(
                    errors
                )
            ),
        ),
    )
    .reset_index()
    .sort_values(
        "price_date"
    )
)

display(
    baseline_daily_metrics.head()
)

display(
    baseline_daily_metrics[
        [
            "weighted_mse_total_variance",
            "unweighted_mse_total_variance",
            "rmse_total_variance",
            "rmse_iv_percentage_points",
            "mean_iv_error_pp",
            "maximum_absolute_iv_error_pp",
        ]
    ].describe()
)

baseline_parameters = (
    baseline_parameters
    .merge(
        baseline_daily_metrics,
        on=[
            "price_date",
            "option_expiry",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
#worst-fitting dates
worst_fitting_dates = (
    baseline_parameters
    .sort_values(
        "rmse_iv_percentage_points",
        ascending=False,
    )
    [
        [
            "price_date",
            "n_options",
            "weighted_mse_total_variance",
            "unweighted_mse_total_variance",
            "rmse_iv_percentage_points",
            "maximum_absolute_iv_error_pp",
            "box_bound_hit_names",
            "active_constraint_names",
        ]
    ]
    .head(10)
)

display(
    worst_fitting_dates
)

worst_weighted_objective_dates = (
    baseline_parameters
    .sort_values(
        "weighted_mse_total_variance",
        ascending=False,
    )
    [
        [
            "price_date",
            "n_options",
            "weighted_mse_total_variance",
            "unweighted_mse_total_variance",
            "rmse_iv_percentage_points",
            "box_bound_hit_names",
            "active_constraint_names",
        ]
    ]
    .head(10)
)

display(
    worst_weighted_objective_dates
)

Even the worst results aren't that bad!
Overall IV RMSE = 0.754 volatility percentage points.
Daily results are also stable (daily bias = -0.0356 percentage points)

In [ ]:
#raw parameter scales
raw_parameter_scale_records = []

for parameter in SVI_PARAMETER_NAMES:
    values = baseline_parameters[
        parameter
    ]

    q1 = values.quantile(
        0.25
    )

    q3 = values.quantile(
        0.75
    )

    raw_parameter_scale_records.append(
        {
            "parameter": parameter,
            "mean": values.mean(),
            "standard_deviation": (
                values.std(
                    ddof=1
                )
            ),
            "q1": q1,
            "median": values.median(),
            "q3": q3,
            "iqr": q3 - q1,
            "minimum": values.min(),
            "maximum": values.max(),
        }
    )

raw_parameter_scales = (
    pd.DataFrame(
        raw_parameter_scale_records
    )
    .set_index(
        "parameter"
    )
)

display(
    raw_parameter_scales
)

In [ ]:
#derived curve-quantity scales
CURVE_QUANTITY_NAMES = [
    "w_min",
    "minimum_location_k",
    "left_asymptotic_slope",
    "right_asymptotic_slope",
]

curve_quantity_scale_records = []

for quantity in CURVE_QUANTITY_NAMES:
    values = baseline_parameters[
        quantity
    ]

    q1 = values.quantile(
        0.25
    )

    q3 = values.quantile(
        0.75
    )

    curve_quantity_scale_records.append(
        {
            "quantity": quantity,
            "mean": values.mean(),
            "standard_deviation": (
                values.std(
                    ddof=1
                )
            ),
            "q1": q1,
            "median": values.median(),
            "q3": q3,
            "iqr": q3 - q1,
            "minimum": values.min(),
            "maximum": values.max(),
        }
    )

curve_quantity_scales = (
    pd.DataFrame(
        curve_quantity_scale_records
    )
    .set_index(
        "quantity"
    )
)

display(
    curve_quantity_scales
)

As expected, right_asymptotic_slope has essentially zero IQR, so don't use that as a penalty scale.
b and rho also have much smaller IQRs than SDs since most dates lie near the right-slope constraint.

In [ ]:
#plot SVI parameter paths
# Plot the five reported raw SVI parameters
# through time.

for parameter in SVI_PARAMETER_NAMES:
    plt.figure(figsize=(10, 5))

    plt.plot(
        baseline_parameters[
            "price_date"
        ],
        baseline_parameters[
            parameter
        ],
        marker="o",
        markersize=3,
    )

    plt.xlabel("Observation date")
    plt.ylabel(parameter)

    plt.title(
        f"Independent Weighted Constrained "
        f"SVI Parameter: {parameter}"
    )

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
#optimization parameter paths
# Plot the five parameters used directly
# by the constrained optimizer.

for parameter in SVI_OPT_PARAMETER_NAMES:
    plt.figure(figsize=(10, 5))

    plt.plot(
        baseline_parameters[
            "price_date"
        ],
        baseline_parameters[
            parameter
        ],
        marker="o",
        markersize=3,
    )

    plt.xlabel("Observation date")
    plt.ylabel(parameter)

    plt.title(
        f"Independent Weighted Constrained "
        f"Optimization Parameter: {parameter}"
    )

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
#plot derived curve quantities
# Plot quantities that describe the fitted
# curve more directly.

for quantity in CURVE_QUANTITY_NAMES:
    plt.figure(figsize=(10, 5))

    plt.plot(
        baseline_parameters[
            "price_date"
        ],
        baseline_parameters[
            quantity
        ],
        marker="o",
        markersize=3,
    )

    plt.xlabel("Observation date")
    plt.ylabel(quantity)

    plt.title(
        f"Independent Weighted Constrained "
        f"SVI Curve Quantity: {quantity}"
    )

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
#daily iv rmse
plt.figure(figsize=(10, 5))

plt.plot(
    baseline_daily_metrics[
        "price_date"
    ],
    baseline_daily_metrics[
        "rmse_iv_percentage_points"
    ],
    marker="o",
    markersize=3,
)

plt.xlabel("Observation date")

plt.ylabel(
    "IV RMSE (percentage points)"
)

plt.title(
    "Independent Weighted Constrained "
    "SVI Daily IV Error"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    baseline_daily_metrics[
        "price_date"
    ],
    baseline_daily_metrics[
        "weighted_mse_total_variance"
    ],
    marker="o",
    markersize=3,
)

plt.xlabel("Observation date")

plt.ylabel(
    "Weighted total-variance MSE"
)

plt.title(
    "Independent Weighted Constrained "
    "SVI Daily Objective"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    baseline_daily_metrics[
        "price_date"
    ],
    baseline_daily_metrics[
        "rmse_total_variance"
    ],
    marker="o",
    markersize=3,
)

plt.xlabel("Observation date")
plt.ylabel("Total-variance RMSE")

plt.title(
    "Independent Weighted Constrained "
    "SVI Daily Unweighted Error"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Again, overall, fitted curves more stable than the raw parameters.
Because the right-wing constraint beta(1+rho) = 2, those two parameters are directly linked.
IQR is probably too aggressive for beta and rho.
a is hard to interpret since it's derived and w_min is smoother.
m generally rises later, svi_sigma declines into March, then rises into May.
w_min has a clear downward trend, and the derived minimum location is mostly consistent.
the left slope is already super stable.
will regularize the five parameters (not a, replace with w_min), definitely not the derived quantities. SD is better than IQR for the standardization.

Overall, the daily SVI curves fit well and the economically meaningful curve quantities evolve fairly coherently, but the raw parameterization occasionally switches between compensating solutions (especially when the right-wing constraint activation switches), so this is exactly why temporal regularization is worthwhile.

In [ ]:
#define ridge-standardization scales
# Parameters that will enter the temporal
# ridge penalty:
#
# (w_min, b, rho, m, svi_sigma)

RIDGE_SCALE_FLOOR_FRACTION = 0.02

ridge_scale_records = []

for parameter in SVI_OPT_PARAMETER_NAMES:
    values = (
        baseline_parameters[
            parameter
        ]
        .astype(float)
    )

    standard_deviation = (
        values.std(ddof=1)
    )

    median_absolute_level = abs(
        values.median()
    )

    # Safety floor to prevent division by a
    # zero or extremely small scale.
    scale_floor = max(
        1e-8,
        RIDGE_SCALE_FLOOR_FRACTION
        * max(
            median_absolute_level,
            1e-3,
        ),
    )

    ridge_scale = max(
        standard_deviation,
        scale_floor,
    )

    ridge_scale_records.append(
        {
            "parameter": parameter,
            "standard_deviation": (
                standard_deviation
            ),
            "scale_floor": scale_floor,
            "ridge_scale": ridge_scale,
            "floor_was_used": (
                ridge_scale
                > standard_deviation
            ),
        }
    )

ridge_parameter_scales = (
    pd.DataFrame(
        ridge_scale_records
    )
    .set_index("parameter")
)

display(
    ridge_parameter_scales
)

In [ ]:
ridge_scale_array = (
    ridge_parameter_scales
    .loc[
        SVI_OPT_PARAMETER_NAMES,
        "ridge_scale",
    ]
    .to_numpy(
        dtype=float
    )
)

assert np.isfinite(
    ridge_scale_array
).all()

assert (
    ridge_scale_array > 0
).all()

print(
    "Ridge parameter order:",
    SVI_OPT_PARAMETER_NAMES,
)

print(
    "Ridge scales:",
    ridge_scale_array,
)

In [ ]:
#day-to-day parameter jumps
ordered_parameters = (
    baseline_parameters
    .sort_values(
        "price_date"
    )
    .reset_index(drop=True)
)

parameter_jump_records = []

for parameter in SVI_OPT_PARAMETER_NAMES:
    parameter_scale = (
        ridge_parameter_scales.loc[
            parameter,
            "ridge_scale",
        ]
    )

    daily_changes = (
        ordered_parameters[
            parameter
        ]
        .diff()
        .dropna()
    )

    standardized_changes = (
        daily_changes
        / parameter_scale
    )

    parameter_jump_records.append(
        {
            "parameter": parameter,

            "mean_absolute_change": (
                daily_changes
                .abs()
                .mean()
            ),

            "median_absolute_change": (
                daily_changes
                .abs()
                .median()
            ),

            "maximum_absolute_change": (
                daily_changes
                .abs()
                .max()
            ),

            "mean_absolute_standardized_change": (
                standardized_changes
                .abs()
                .mean()
            ),

            "median_absolute_standardized_change": (
                standardized_changes
                .abs()
                .median()
            ),

            "maximum_absolute_standardized_change": (
                standardized_changes
                .abs()
                .max()
            ),
        }
    )

parameter_jump_summary = (
    pd.DataFrame(
        parameter_jump_records
    )
    .set_index("parameter")
)

display(
    parameter_jump_summary
)

In [ ]:
#largest standardized jumps
largest_jump_records = []

for parameter in SVI_OPT_PARAMETER_NAMES:
    parameter_scale = (
        ridge_parameter_scales.loc[
            parameter,
            "ridge_scale",
        ]
    )

    raw_changes = (
        ordered_parameters[
            parameter
        ]
        .diff()
    )

    standardized_changes = (
        raw_changes
        / parameter_scale
    )

    largest_jump_index = (
        standardized_changes
        .abs()
        .idxmax()
    )

    largest_jump_records.append(
        {
            "parameter": parameter,

            "previous_date": (
                ordered_parameters.loc[
                    largest_jump_index - 1,
                    "price_date",
                ]
            ),

            "current_date": (
                ordered_parameters.loc[
                    largest_jump_index,
                    "price_date",
                ]
            ),

            "previous_value": (
                ordered_parameters.loc[
                    largest_jump_index - 1,
                    parameter,
                ]
            ),

            "current_value": (
                ordered_parameters.loc[
                    largest_jump_index,
                    parameter,
                ]
            ),

            "raw_change": (
                raw_changes.loc[
                    largest_jump_index
                ]
            ),

            "standardized_change": (
                standardized_changes.loc[
                    largest_jump_index
                ]
            ),

            "absolute_standardized_change": abs(
                standardized_changes.loc[
                    largest_jump_index
                ]
            ),
        }
    )

largest_parameter_jumps = (
    pd.DataFrame(
        largest_jump_records
    )
    .set_index("parameter")
)

display(
    largest_parameter_jumps
)

In [ ]:
#standardized daily parameter changes
for parameter in SVI_OPT_PARAMETER_NAMES:
    parameter_scale = (
        ridge_parameter_scales.loc[
            parameter,
            "ridge_scale",
        ]
    )

    standardized_changes = (
        ordered_parameters[
            parameter
        ]
        .diff()
        / parameter_scale
    )

    plt.figure(figsize=(10, 5))

    plt.plot(
        ordered_parameters[
            "price_date"
        ],
        standardized_changes,
        marker="o",
        markersize=3,
    )

    plt.axhline(
        0,
        linestyle="--",
        alpha=0.7,
    )

    plt.axhline(
        1,
        linestyle=":",
        alpha=0.5,
    )

    plt.axhline(
        -1,
        linestyle=":",
        alpha=0.5,
    )

    plt.xlabel("Observation date")

    plt.ylabel(
        "Daily change in baseline SDs"
    )

    plt.title(
        f"Standardized Daily Change: "
        f"{parameter}"
    )

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
#completed ind. baseline
baseline_predictions.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_predictions.csv",
    index=False,
)

baseline_parameters.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_parameters.csv",
    index=False,
)

baseline_daily_metrics.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_daily_metrics.csv",
    index=False,
)

raw_parameter_scales.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_raw_parameter_scales.csv"
)

curve_quantity_scales.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_curve_quantity_scales.csv"
)

ridge_parameter_scales.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_ridge_scales.csv"
)

parameter_jump_summary.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_jump_summary.csv"
)

largest_parameter_jumps.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_largest_jumps.csv"
)

preflight_results.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_preflight.csv",
    index=False,
)

overall_metrics.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_overall_metrics.csv"
)

box_bound_summary.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_box_bound_summary.csv"
)

active_constraint_summary.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_active_constraint_summary.csv"
)

individual_box_bound_summary.to_csv(
    OUTPUT_DIR
    / "baseline_weighted_constrained_svi_individual_bound_summary.csv",
    index=False,
)

print(
    "Independent weighted constrained "
    "SVI baseline saved."
)

w_min already fairly stable.
svi_sigma moderate movement
b, rho, m real instability. (again, spikes in b and rho occur together)
simultaneous spike in m, so probably just parameter compensation.

Regularized (Final) Model:
Model - SVI (same)
Selection metric - (MSE of total implied variance) + lambda * (L2 regularization of standardized temporal-neighbor parameters)
     - for the first data point, just use MSE of total implied variance (no predecessor)
Try lambda = {0.0, 10^(-8), 3e-8, 1e-7, 3e-7, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4}
     - given fitting loss is only around 5 x 1e-6
     - for each, completely restart
Keep track of the same things as for the baseline iteration (one chart for each lambda iteration)


In [ ]:
#define sequential ridge objective
def svi_ridge_objective(
    phi,
    k_values,
    market_w,
    previous_phi,
    ridge_scales,
    ridge_lambda,
):
    """
    Weighted SVI fitting loss plus a
    standardized temporal ridge penalty.

    The five optimization parameters are:
        (w_min, b, rho, m, svi_sigma)
    """

    fitting_loss = svi_mse_transformed(
        phi,
        k_values,
        market_w,
    )

    standardized_change = (
        (phi - previous_phi)
        / ridge_scales
    )

    ridge_penalty = np.mean(
        standardized_change**2
    )

    total_objective = (
        fitting_loss
        + ridge_lambda
        * ridge_penalty
    )

    return total_objective

In [ ]:
#starts for one regularized date
#ind. fit for current data, prev. regularized fit, some midpoint
def make_regularized_initial_guesses(
    k_values,
    market_w,
    independent_phi,
    previous_phi,
):
    """
    Create feasible deterministic starts for
    one sequential regularized calibration.
    """

    candidate_starts = [
        np.asarray(
            independent_phi,
            dtype=float,
        ),

        np.asarray(
            previous_phi,
            dtype=float,
        ),

        0.5
        * (
            np.asarray(
                independent_phi,
                dtype=float,
            )
            + np.asarray(
                previous_phi,
                dtype=float,
            )
        ),
    ]

    candidate_starts.extend(
        make_svi_initial_guesses(
            k_values,
            market_w,
        )
    )

    feasible_starts = []

    for candidate in candidate_starts:
        candidate = np.asarray(
            candidate,
            dtype=float,
        ).copy()

        # Keep every coordinate inside its
        # box constraints.
        for index, (
            lower_bound,
            upper_bound,
        ) in enumerate(SVI_OPT_BOUNDS):
            candidate[index] = np.clip(
                candidate[index],
                lower_bound + 1e-10,
                upper_bound - 1e-10,
            )

        # Enforce the two wing-slope restrictions
        # by reducing b if necessary.
        rho = candidate[2]

        maximum_b = min(
            WING_SLOPE_LIMIT
            / max(
                1.0 - rho,
                1e-8,
            ),
            WING_SLOPE_LIMIT
            / max(
                1.0 + rho,
                1e-8,
            ),
            SVI_OPT_BOUNDS[1][1],
        )

        candidate[1] = min(
            candidate[1],
            maximum_b
            * (1.0 - 1e-10),
        )

        feasible_starts.append(
            candidate
        )

    # Remove nearly duplicate starts.
    unique_starts = []

    for candidate in feasible_starts:
        is_duplicate = any(
            np.allclose(
                candidate,
                existing,
                rtol=1e-8,
                atol=1e-10,
            )
            for existing in unique_starts
        )

        if not is_duplicate:
            unique_starts.append(
                candidate
            )

    return unique_starts

In [ ]:
#fit one date
def fit_regularized_svi_snapshot(
    k_values,
    market_w,
    previous_phi,
    independent_phi,
    ridge_scales,
    ridge_lambda,
):
    """
    Fit one constrained SVI curve with a
    standardized penalty relative to the
    previous regularized parameter vector.
    """

    k_values = np.asarray(
        k_values,
        dtype=float,
    )

    market_w = np.asarray(
        market_w,
        dtype=float,
    )

    previous_phi = np.asarray(
        previous_phi,
        dtype=float,
    )

    independent_phi = np.asarray(
        independent_phi,
        dtype=float,
    )

    ridge_scales = np.asarray(
        ridge_scales,
        dtype=float,
    )

    initial_guesses = (
        make_regularized_initial_guesses(
            k_values,
            market_w,
            independent_phi,
            previous_phi,
        )
    )

    weights, variance_floor = (
        make_total_variance_weights(
            market_w
        )
    )

    wing_slope_constraints = [
        {
            "type": "ineq",
            "fun": lambda phi: (
                WING_SLOPE_LIMIT
                - phi[1]
                * (1.0 - phi[2])
            ),
        },
        {
            "type": "ineq",
            "fun": lambda phi: (
                WING_SLOPE_LIMIT
                - phi[1]
                * (1.0 + phi[2])
            ),
        },
    ]

    candidates = []

    for start_index, initial_guess in enumerate(
        initial_guesses
    ):
        result = minimize(
            fun=svi_ridge_objective,
            x0=initial_guess,
            args=(
                k_values,
                market_w,
                previous_phi,
                ridge_scales,
                ridge_lambda,
            ),
            method="SLSQP",
            bounds=SVI_OPT_BOUNDS,
            constraints=wing_slope_constraints,
            options={
                "maxiter": 3000,
                "ftol": 1e-12,
                "disp": False,
            },
        )

        phi = np.asarray(
            result.x,
            dtype=float,
        )

        raw_theta = (
            optimization_to_raw_theta(
                phi
            )
        )

        predicted_w = raw_svi(
            k_values,
            *raw_theta,
        )

        (
            left_slope,
            right_slope,
        ) = svi_asymptotic_slopes(
            raw_theta
        )

        valid = (
            np.isfinite(phi).all()
            and np.isfinite(
                raw_theta
            ).all()
            and np.isfinite(
                predicted_w
            ).all()
            and np.all(
                predicted_w > 0
            )
            and phi[0] > 0
            and (
                left_slope
                <= WING_SLOPE_LIMIT
                + CONSTRAINT_TOLERANCE
            )
            and (
                right_slope
                <= WING_SLOPE_LIMIT
                + CONSTRAINT_TOLERANCE
            )
        )

        if not valid:
            continue

        errors = (
            predicted_w
            - market_w
        )

        weighted_mse = np.average(
            errors**2,
            weights=weights,
        )

        unweighted_mse = np.mean(
            errors**2
        )

        standardized_change = (
            (phi - previous_phi)
            / ridge_scales
        )

        ridge_penalty = np.mean(
            standardized_change**2
        )

        total_objective = (
            weighted_mse
            + ridge_lambda
            * ridge_penalty
        )

        candidates.append(
            {
                "result": result,
                "phi": phi,
                "theta": raw_theta,
                "weighted_mse": (
                    weighted_mse
                ),
                "unweighted_mse": (
                    unweighted_mse
                ),
                "ridge_penalty": (
                    ridge_penalty
                ),
                "total_objective": (
                    total_objective
                ),
                "start_index": (
                    start_index
                ),
            }
        )

    if not candidates:
        raise RuntimeError(
            "No valid regularized SVI "
            "candidate was found."
        )

    successful_candidates = [
        candidate
        for candidate in candidates
        if candidate[
            "result"
        ].success
    ]

    candidate_pool = (
        successful_candidates
        if successful_candidates
        else candidates
    )

    best_candidate = min(
        candidate_pool,
        key=lambda candidate:
        candidate[
            "total_objective"
        ],
    )

    best_result = best_candidate[
        "result"
    ]

    best_phi = best_candidate[
        "phi"
    ]

    best_theta = best_candidate[
        "theta"
    ]

    (
        left_slope,
        right_slope,
    ) = svi_asymptotic_slopes(
        best_theta
    )

    box_bound_hits = (
        identify_box_bound_hits(
            best_phi,
            SVI_OPT_PARAMETER_NAMES,
            SVI_OPT_BOUNDS,
        )
    )

    active_constraints = (
        identify_active_slope_constraints(
            left_slope,
            right_slope,
        )
    )

    return {
        "theta": best_theta,
        "optimization_theta": best_phi,

        "weighted_mse": best_candidate[
            "weighted_mse"
        ],

        "unweighted_mse": best_candidate[
            "unweighted_mse"
        ],

        "ridge_penalty": best_candidate[
            "ridge_penalty"
        ],

        "total_objective": best_candidate[
            "total_objective"
        ],

        "weight_variance_floor": (
            variance_floor
        ),

        "optimizer_success": bool(
            best_result.success
        ),

        "optimizer_message": str(
            best_result.message
        ),

        "n_iterations": (
            best_result.nit
        ),

        "n_function_evaluations": (
            best_result.nfev
        ),

        "best_start_index": (
            best_candidate[
                "start_index"
            ]
        ),

        "n_initial_starts": (
            len(initial_guesses)
        ),

        "n_valid_starts": (
            len(candidates)
        ),

        "n_successful_starts": (
            len(
                successful_candidates
            )
        ),

        "global_minimum_w": (
            best_phi[0]
        ),

        "minimum_location_k": (
            svi_minimum_location(
                best_theta
            )
        ),

        "left_asymptotic_slope": (
            left_slope
        ),

        "right_asymptotic_slope": (
            right_slope
        ),

        "box_bound_hit_names": (
            box_bound_hits
        ),

        "active_constraint_names": (
            active_constraints
        ),

        "n_box_bound_hits": (
            len(box_bound_hits)
        ),

        "n_active_constraints": (
            len(active_constraints)
        ),
    }

In [ ]:
RIDGE_LAMBDA_GRID = [
    0.0,
    1e-8,
    3e-8,
    1e-7,
    3e-7,
    1e-6,
    3e-6,
    1e-5,
    3e-5,
    1e-4,
]


def format_lambda(ridge_lambda):
    if ridge_lambda == 0:
        return "0"

    return f"{ridge_lambda:.0e}"

In [ ]:
#independent fit lookup (for finding starting points)
baseline_parameter_lookup = (
    baseline_parameters
    .set_index("price_date")
    .sort_index()
)

assert (
    baseline_parameter_lookup.index.is_unique
)

assert set(
    SVI_OPT_PARAMETER_NAMES
).issubset(
    baseline_parameter_lookup.columns
)

print(
    "Independent parameter lookup ready."
)

In [ ]:
#function for one complete lambda path
def run_regularized_svi_path(
    ridge_lambda,
):
    """
    Run one complete sequential 103-date
    SVI calibration path for a fixed lambda.

    Each lambda starts again from the first
    date and uses its own previous regularized
    parameter vector.
    """

    parameter_records = []
    prediction_frames = []

    previous_phi = None

    for date_index, selected_date in enumerate(
        available_dates
    ):
        (
            k_values,
            market_w,
            time_to_expiry,
            snapshot,
        ) = get_snapshot_data(
            model_data,
            selected_date,
        )

        independent_phi = (
            baseline_parameter_lookup
            .loc[
                selected_date,
                SVI_OPT_PARAMETER_NAMES,
            ]
            .to_numpy(
                dtype=float
            )
        )

        # The first date has no predecessor,
        # so it uses fitting loss only.
        if date_index == 0:
            fit = fit_svi_snapshot(
                k_values,
                market_w,
            )

            phi = np.asarray(
                fit[
                    "optimization_theta"
                ],
                dtype=float,
            )

            standardized_change = (
                np.full(
                    len(
                        SVI_OPT_PARAMETER_NAMES
                    ),
                    np.nan,
                )
            )

            ridge_penalty = 0.0

            total_objective = fit[
                "weighted_mse"
            ]

        else:
            fit = (
                fit_regularized_svi_snapshot(
                    k_values=k_values,
                    market_w=market_w,
                    previous_phi=previous_phi,
                    independent_phi=(
                        independent_phi
                    ),
                    ridge_scales=(
                        ridge_scale_array
                    ),
                    ridge_lambda=(
                        ridge_lambda
                    ),
                )
            )

            phi = np.asarray(
                fit[
                    "optimization_theta"
                ],
                dtype=float,
            )

            standardized_change = (
                (phi - previous_phi)
                / ridge_scale_array
            )

            ridge_penalty = fit[
                "ridge_penalty"
            ]

            total_objective = fit[
                "total_objective"
            ]

        theta = np.asarray(
            fit["theta"],
            dtype=float,
        )

        predicted_w = raw_svi(
            k_values,
            *theta,
        )

        predicted_iv = np.sqrt(
            predicted_w
            / time_to_expiry
        )

        weights, variance_floor = (
            make_total_variance_weights(
                market_w
            )
        )

        current_predictions = (
            snapshot.copy()
        )

        current_predictions[
            "ridge_lambda"
        ] = ridge_lambda

        current_predictions[
            "predicted_total_variance"
        ] = predicted_w

        current_predictions[
            "total_variance_error"
        ] = (
            predicted_w
            - market_w
        )

        current_predictions[
            "squared_total_variance_error"
        ] = (
            current_predictions[
                "total_variance_error"
            ] ** 2
        )

        current_predictions[
            "total_variance_weight"
        ] = weights

        current_predictions[
            "weighted_squared_total_variance_error"
        ] = (
            weights
            * current_predictions[
                "squared_total_variance_error"
            ]
        )

        current_predictions[
            "predicted_iv"
        ] = predicted_iv

        current_predictions[
            "iv_error"
        ] = (
            predicted_iv
            - current_predictions[
                "iv_decimal"
            ]
        )

        current_predictions[
            "iv_error_pp"
        ] = (
            100.0
            * current_predictions[
                "iv_error"
            ]
        )

        current_predictions[
            "squared_iv_error"
        ] = (
            current_predictions[
                "iv_error"
            ] ** 2
        )

        current_predictions[
            "weight_variance_floor"
        ] = variance_floor

        prediction_frames.append(
            current_predictions
        )

        parameter_record = {
            "ridge_lambda": (
                ridge_lambda
            ),

            "price_date": (
                selected_date
            ),

            "option_expiry": snapshot[
                "option_expiry"
            ].iloc[0],

            "n_options": len(
                snapshot
            ),

            # Raw SVI parameters
            "a": theta[0],
            "b": theta[1],
            "rho": theta[2],
            "m": theta[3],
            "svi_sigma": theta[4],

            # Optimization parameters
            "w_min": phi[0],

            # Derived curve quantities
            "minimum_location_k": fit[
                "minimum_location_k"
            ],

            "left_asymptotic_slope": fit[
                "left_asymptotic_slope"
            ],

            "right_asymptotic_slope": fit[
                "right_asymptotic_slope"
            ],

            # Fitting terms
            "weighted_mse": fit[
                "weighted_mse"
            ],

            "unweighted_mse": fit[
                "unweighted_mse"
            ],

            "ridge_penalty": (
                ridge_penalty
            ),

            "total_objective": (
                total_objective
            ),

            "weight_variance_floor": (
                variance_floor
            ),

            # Optimizer diagnostics
            "optimizer_success": fit[
                "optimizer_success"
            ],

            "optimizer_message": fit[
                "optimizer_message"
            ],

            "n_iterations": fit[
                "n_iterations"
            ],

            "n_function_evaluations": fit[
                "n_function_evaluations"
            ],

            "n_initial_starts": fit[
                "n_initial_starts"
            ],

            "n_valid_starts": fit[
                "n_valid_starts"
            ],

            "n_successful_starts": fit[
                "n_successful_starts"
            ],

            "best_start_index": fit[
                "best_start_index"
            ],

            # Constraint diagnostics
            "n_box_bound_hits": fit[
                "n_box_bound_hits"
            ],

            "box_bound_hit_names": (
                ", ".join(
                    fit[
                        "box_bound_hit_names"
                    ]
                )
            ),

            "n_active_constraints": fit[
                "n_active_constraints"
            ],

            "active_constraint_names": (
                ", ".join(
                    fit[
                        "active_constraint_names"
                    ]
                )
            ),
        }

        for parameter_index, parameter in enumerate(
            SVI_OPT_PARAMETER_NAMES
        ):
            parameter_record[
                f"standardized_change_{parameter}"
            ] = standardized_change[
                parameter_index
            ]

        if date_index == 0:
            parameter_record[
                "mean_absolute_standardized_change"
            ] = np.nan

            parameter_record[
                "maximum_absolute_standardized_change"
            ] = np.nan

        else:
            parameter_record[
                "mean_absolute_standardized_change"
            ] = np.mean(
                np.abs(
                    standardized_change
                )
            )

            parameter_record[
                "maximum_absolute_standardized_change"
            ] = np.max(
                np.abs(
                    standardized_change
                )
            )

        parameter_records.append(
            parameter_record
        )

        # This lambda's next date uses this
        # lambda's current fitted parameters.
        previous_phi = phi.copy()

    path_predictions = (
        pd.concat(
            prediction_frames,
            ignore_index=True,
        )
        .sort_values(
            [
                "price_date",
                "log_moneyness",
            ]
        )
        .reset_index(drop=True)
    )

    path_parameters = (
        pd.DataFrame(
            parameter_records
        )
        .sort_values(
            "price_date"
        )
        .reset_index(drop=True)
    )

    return {
        "ridge_lambda": ridge_lambda,
        "predictions": (
            path_predictions
        ),
        "parameters": (
            path_parameters
        ),
    }

In [ ]:
#smoke test
# Confirm ridge scales are usable

print(
    pd.Series(
        ridge_scale_array,
        index=SVI_OPT_PARAMETER_NAMES,
        name="ridge_scale",
    )
)

assert len(ridge_scale_array) == 5
assert np.isfinite(ridge_scale_array).all()
assert (ridge_scale_array > 0).all()

print("Ridge scales passed checks.")

In [ ]:
#first date
smoke_first_date = available_dates[0]
smoke_second_date = available_dates[1]

(
    k_first_smoke,
    w_first_smoke,
    T_first_smoke,
    snapshot_first_smoke,
) = get_snapshot_data(
    model_data,
    smoke_first_date,
)

first_smoke_fit = fit_svi_snapshot(
    k_first_smoke,
    w_first_smoke,
)

previous_phi_smoke = np.asarray(
    first_smoke_fit[
        "optimization_theta"
    ],
    dtype=float,
)

print(
    "First date:",
    smoke_first_date,
)

print(
    "First-date optimizer success:",
    first_smoke_fit[
        "optimizer_success"
    ],
)

print(
    "First-date phi:",
    previous_phi_smoke,
)

assert first_smoke_fit[
    "optimizer_success"
]

assert np.isfinite(
    previous_phi_smoke
).all()

In [ ]:
#second date, lambda 0
(
    k_second_smoke,
    w_second_smoke,
    T_second_smoke,
    snapshot_second_smoke,
) = get_snapshot_data(
    model_data,
    smoke_second_date,
)

second_independent_fit = (
    fit_svi_snapshot(
        k_second_smoke,
        w_second_smoke,
    )
)

second_independent_phi = np.asarray(
    second_independent_fit[
        "optimization_theta"
    ],
    dtype=float,
)

second_lambda_zero_fit = (
    fit_regularized_svi_snapshot(
        k_values=k_second_smoke,
        market_w=w_second_smoke,
        previous_phi=previous_phi_smoke,
        independent_phi=(
            second_independent_phi
        ),
        ridge_scales=(
            ridge_scale_array
        ),
        ridge_lambda=0.0,
    )
)

print(
    "Lambda-zero success:",
    second_lambda_zero_fit[
        "optimizer_success"
    ],
)

print(
    "Independent weighted MSE:",
    second_independent_fit[
        "weighted_mse"
    ],
)

print(
    "Lambda-zero weighted MSE:",
    second_lambda_zero_fit[
        "weighted_mse"
    ],
)

In [ ]:
#compare fitted curves
independent_w_smoke = raw_svi(
    k_second_smoke,
    *second_independent_fit[
        "theta"
    ],
)

lambda_zero_w_smoke = raw_svi(
    k_second_smoke,
    *second_lambda_zero_fit[
        "theta"
    ],
)

maximum_lambda_zero_w_difference = (
    np.max(
        np.abs(
            independent_w_smoke
            - lambda_zero_w_smoke
        )
    )
)

print(
    "Maximum fitted-w difference:",
    maximum_lambda_zero_w_difference,
)

assert second_lambda_zero_fit[
    "optimizer_success"
]

assert (
    lambda_zero_w_smoke > 0
).all()

In [ ]:
print(
    "Independent weighted MSE:",
    second_independent_fit[
        "weighted_mse"
    ],
)

print(
    "Lambda-zero weighted MSE:",
    second_lambda_zero_fit[
        "weighted_mse"
    ],
)

print(
    "Weighted-MSE difference:",
    abs(
        second_independent_fit[
            "weighted_mse"
        ]
        - second_lambda_zero_fit[
            "weighted_mse"
        ]
    ),
)

In [ ]:
#compare lambda 0 objective values
print(
    "Independent weighted MSE:",
    second_independent_fit[
        "weighted_mse"
    ],
)

print(
    "Lambda-zero weighted MSE:",
    second_lambda_zero_fit[
        "weighted_mse"
    ],
)

weighted_mse_difference = abs(
    second_independent_fit[
        "weighted_mse"
    ]
    - second_lambda_zero_fit[
        "weighted_mse"
    ]
)

print(
    "Weighted-MSE difference:",
    weighted_mse_difference,
)

assert np.isclose(
    second_independent_fit[
        "weighted_mse"
    ],
    second_lambda_zero_fit[
        "weighted_mse"
    ],
    rtol=1e-10,
    atol=1e-14,
)

print(
    "Lambda zero exactly reproduces "
    "the independent fit."
)

In [ ]:
#test one nonzero lambda
SMOKE_TEST_LAMBDA = 1e-6

second_regularized_smoke_fit = (
    fit_regularized_svi_snapshot(
        k_values=k_second_smoke,
        market_w=w_second_smoke,
        previous_phi=previous_phi_smoke,
        independent_phi=(
            second_independent_phi
        ),
        ridge_scales=(
            ridge_scale_array
        ),
        ridge_lambda=(
            SMOKE_TEST_LAMBDA
        ),
    )
)

regularized_phi_smoke = np.asarray(
    second_regularized_smoke_fit[
        "optimization_theta"
    ],
    dtype=float,
)

regularized_w_smoke = raw_svi(
    k_second_smoke,
    *second_regularized_smoke_fit[
        "theta"
    ],
)

print(
    "Optimizer success:",
    second_regularized_smoke_fit[
        "optimizer_success"
    ],
)

print(
    "Weighted MSE:",
    second_regularized_smoke_fit[
        "weighted_mse"
    ],
)

print(
    "Ridge penalty:",
    second_regularized_smoke_fit[
        "ridge_penalty"
    ],
)

print(
    "Lambda times ridge penalty:",
    SMOKE_TEST_LAMBDA
    * second_regularized_smoke_fit[
        "ridge_penalty"
    ],
)

print(
    "Total objective:",
    second_regularized_smoke_fit[
        "total_objective"
    ],
)

print(
    "Box-bound hits:",
    second_regularized_smoke_fit[
        "box_bound_hit_names"
    ],
)

print(
    "Active constraints:",
    second_regularized_smoke_fit[
        "active_constraint_names"
    ],
)

assert second_regularized_smoke_fit[
    "optimizer_success"
]

assert np.isfinite(
    regularized_phi_smoke
).all()

assert np.isfinite(
    regularized_w_smoke
).all()

assert (
    regularized_w_smoke > 0
).all()

assert np.isfinite(
    second_regularized_smoke_fit[
        "ridge_penalty"
    ]
)

assert (
    second_regularized_smoke_fit[
        "ridge_penalty"
    ] >= 0
)

assert np.isclose(
    second_regularized_smoke_fit[
        "total_objective"
    ],
    (
        second_regularized_smoke_fit[
            "weighted_mse"
        ]
        + SMOKE_TEST_LAMBDA
        * second_regularized_smoke_fit[
            "ridge_penalty"
        ]
    ),
    rtol=1e-9,
    atol=1e-14,
)

assert (
    second_regularized_smoke_fit[
        "left_asymptotic_slope"
    ]
    <= WING_SLOPE_LIMIT
    + CONSTRAINT_TOLERANCE
)

assert (
    second_regularized_smoke_fit[
        "right_asymptotic_slope"
    ]
    <= WING_SLOPE_LIMIT
    + CONSTRAINT_TOLERANCE
)

print(
    "Nonzero-lambda smoke test passed."
)

In [ ]:
#running all ten lambda paths
import time

# Start from an empty dictionary so every
# lambda path is completely restarted.
regularized_results = {}

total_grid_start_time = time.time()

for lambda_index, ridge_lambda in enumerate(
    RIDGE_LAMBDA_GRID,
    start=1,
):
    lambda_start_time = time.time()

    print(
        "\n"
        + "=" * 60
    )

    print(
        "Starting lambda "
        f"{format_lambda(ridge_lambda)} "
        f"({lambda_index} / "
        f"{len(RIDGE_LAMBDA_GRID)})"
    )

    try:
        path_result = (
            run_regularized_svi_path(
                ridge_lambda
            )
        )

    except Exception as error:
        print(
            "Lambda path failed:",
            format_lambda(
                ridge_lambda
            ),
        )

        raise error

    regularized_results[
        ridge_lambda
    ] = path_result

    path_parameters = path_result[
        "parameters"
    ]

    path_predictions = path_result[
        "predictions"
    ]

    lambda_elapsed_seconds = (
        time.time()
        - lambda_start_time
    )

    print(
        "Completed lambda:",
        format_lambda(
            ridge_lambda
        ),
    )

    print(
        "Optimizer successes:",
        int(
            path_parameters[
                "optimizer_success"
            ].sum()
        ),
        "/",
        len(path_parameters),
    )

    print(
        "Valid prediction rows:",
        len(path_predictions),
    )

    print(
        "Dates with box-bound hits:",
        int(
            (
                path_parameters[
                    "n_box_bound_hits"
                ] > 0
            ).sum()
        ),
    )

    print(
        "Elapsed minutes:",
        round(
            lambda_elapsed_seconds
            / 60.0,
            2,
        ),
    )

total_grid_elapsed_seconds = (
    time.time()
    - total_grid_start_time
)

print(
    "\n"
    + "=" * 60
)

print(
    "All lambda paths completed."
)

print(
    "Total elapsed minutes:",
    round(
        total_grid_elapsed_seconds
        / 60.0,
        2,
    ),
)

In [ ]:
print(
    "Completed lambdas:",
    list(
        regularized_results.keys()
    ),
)

assert (
    len(regularized_results)
    == len(RIDGE_LAMBDA_GRID)
)

In [ ]:
for ridge_lambda in RIDGE_LAMBDA_GRID:
    result = regularized_results[
        ridge_lambda
    ]

    predictions = result[
        "predictions"
    ]

    parameters = result[
        "parameters"
    ]

    # Correct number of observations and dates
    assert (
        len(predictions)
        == len(model_data)
    )

    assert (
        len(parameters)
        == len(available_dates)
    )

    assert (
        parameters["price_date"].nunique()
        == len(available_dates)
    )

    # All optimizations succeeded
    assert parameters[
        "optimizer_success"
    ].all()

    assert (
        parameters[
            "n_valid_starts"
        ] > 0
    ).all()

    # Predicted quantities are finite and positive
    assert predictions[
        "predicted_total_variance"
    ].notna().all()

    assert (
        predictions[
            "predicted_total_variance"
        ] > 0
    ).all()

    assert predictions[
        "predicted_iv"
    ].notna().all()

    assert (
        predictions[
            "predicted_iv"
        ] > 0
    ).all()

    assert (
        parameters["w_min"] > 0
    ).all()

    # Wing restrictions are satisfied
    assert (
        parameters[
            "left_asymptotic_slope"
        ]
        <= WING_SLOPE_LIMIT
        + CONSTRAINT_TOLERANCE
    ).all()

    assert (
        parameters[
            "right_asymptotic_slope"
        ]
        <= WING_SLOPE_LIMIT
        + CONSTRAINT_TOLERANCE
    ).all()

    # First date has no predecessor
    assert np.isclose(
        parameters.loc[
            0,
            "ridge_penalty",
        ],
        0.0,
    )

    assert np.isclose(
        parameters.loc[
            0,
            "total_objective",
        ],
        parameters.loc[
            0,
            "weighted_mse",
        ],
        rtol=1e-9,
        atol=1e-14,
    )

    # Later dates satisfy:
    # total = fitting loss + lambda * penalty
    expected_objective = (
        parameters.loc[
            1:,
            "weighted_mse",
        ]
        + ridge_lambda
        * parameters.loc[
            1:,
            "ridge_penalty",
        ]
    )

    assert np.allclose(
        parameters.loc[
            1:,
            "total_objective",
        ],
        expected_objective,
        rtol=1e-8,
        atol=1e-13,
    )

    print(
        "Passed:",
        format_lambda(
            ridge_lambda
        ),
    )

print(
    "All regularized paths "
    "passed validation."
)

In [ ]:
#combine lambda results
regularized_all_predictions = (
    pd.concat(
        [
            regularized_results[
                ridge_lambda
            ]["predictions"]
            for ridge_lambda
            in RIDGE_LAMBDA_GRID
        ],
        ignore_index=True,
    )
)

regularized_all_parameters = (
    pd.concat(
        [
            regularized_results[
                ridge_lambda
            ]["parameters"]
            for ridge_lambda
            in RIDGE_LAMBDA_GRID
        ],
        ignore_index=True,
    )
)

expected_prediction_rows = (
    len(model_data)
    * len(RIDGE_LAMBDA_GRID)
)

expected_parameter_rows = (
    len(available_dates)
    * len(RIDGE_LAMBDA_GRID)
)

print(
    "Combined prediction rows:",
    len(
        regularized_all_predictions
    ),
)

print(
    "Expected prediction rows:",
    expected_prediction_rows,
)

print(
    "Combined parameter rows:",
    len(
        regularized_all_parameters
    ),
)

print(
    "Expected parameter rows:",
    expected_parameter_rows,
)

assert (
    len(
        regularized_all_predictions
    )
    == expected_prediction_rows
)

assert (
    len(
        regularized_all_parameters
    )
    == expected_parameter_rows
)

In [ ]:
lambda_zero_predictions = (
    regularized_results[
        0.0
    ]["predictions"]
    .sort_values(
        [
            "price_date",
            "log_moneyness",
        ]
    )
    .reset_index(drop=True)
)

ordered_baseline_predictions = (
    baseline_predictions
    .sort_values(
        [
            "price_date",
            "log_moneyness",
        ]
    )
    .reset_index(drop=True)
)

assert (
    len(lambda_zero_predictions)
    == len(ordered_baseline_predictions)
)

# Confirm that the same observations are
# aligned before comparing predictions.
assert (
    lambda_zero_predictions[
        "price_date"
    ]
    .equals(
        ordered_baseline_predictions[
            "price_date"
        ]
    )
)

assert np.allclose(
    lambda_zero_predictions[
        "log_moneyness"
    ].to_numpy(),
    ordered_baseline_predictions[
        "log_moneyness"
    ].to_numpy(),
    rtol=0.0,
    atol=1e-14,
)

total_variance_differences = (
    lambda_zero_predictions[
        "predicted_total_variance"
    ].to_numpy()
    -
    ordered_baseline_predictions[
        "predicted_total_variance"
    ].to_numpy()
)

iv_difference_pp = (
    100.0
    * (
        lambda_zero_predictions[
            "predicted_iv"
        ].to_numpy()
        -
        ordered_baseline_predictions[
            "predicted_iv"
        ].to_numpy()
    )
)

maximum_w_difference = np.max(
    np.abs(
        total_variance_differences
    )
)

rmse_w_difference = np.sqrt(
    np.mean(
        total_variance_differences**2
    )
)

maximum_iv_difference_pp = np.max(
    np.abs(
        iv_difference_pp
    )
)

rmse_iv_difference_pp = np.sqrt(
    np.mean(
        iv_difference_pp**2
    )
)

baseline_weighted_mse = (
    ordered_baseline_predictions[
        "weighted_squared_total_variance_error"
    ].mean()
)

lambda_zero_weighted_mse = (
    lambda_zero_predictions[
        "weighted_squared_total_variance_error"
    ].mean()
)

weighted_mse_relative_difference_percent = (
    100.0
    * abs(
        lambda_zero_weighted_mse
        - baseline_weighted_mse
    )
    / baseline_weighted_mse
)

baseline_iv_rmse_pp = (
    100.0
    * np.sqrt(
        ordered_baseline_predictions[
            "squared_iv_error"
        ].mean()
    )
)

lambda_zero_iv_rmse_pp = (
    100.0
    * np.sqrt(
        lambda_zero_predictions[
            "squared_iv_error"
        ].mean()
    )
)

print(
    "Maximum total-variance difference:",
    maximum_w_difference,
)

print(
    "RMSE of total-variance differences:",
    rmse_w_difference,
)

print(
    "Maximum IV difference "
    "(percentage points):",
    maximum_iv_difference_pp,
)

print(
    "RMSE of IV differences "
    "(percentage points):",
    rmse_iv_difference_pp,
)

print(
    "Baseline weighted MSE:",
    baseline_weighted_mse,
)

print(
    "Lambda-zero weighted MSE:",
    lambda_zero_weighted_mse,
)

print(
    "Weighted-MSE relative difference (%):",
    weighted_mse_relative_difference_percent,
)

print(
    "Baseline IV RMSE (pp):",
    baseline_iv_rmse_pp,
)

print(
    "Lambda-zero IV RMSE (pp):",
    lambda_zero_iv_rmse_pp,
)

In [ ]:
#lambda summary table
lambda_summary_records = []

standardized_change_columns = [
    f"standardized_change_{parameter}"
    for parameter in SVI_OPT_PARAMETER_NAMES
]

for ridge_lambda in RIDGE_LAMBDA_GRID:
    predictions = (
        regularized_results[
            ridge_lambda
        ]["predictions"]
    )

    parameters = (
        regularized_results[
            ridge_lambda
        ]["parameters"]
    )

    # Exclude the first date because it
    # does not have a predecessor.
    standardized_changes = (
        parameters[
            standardized_change_columns
        ]
        .iloc[1:]
        .to_numpy(
            dtype=float
        )
    )

    assert np.isfinite(
        standardized_changes
    ).all()

    daily_metrics = (
        predictions
        .groupby(
            "price_date",
            as_index=False,
        )
        .agg(
            daily_weighted_mse=(
                "weighted_squared_total_variance_error",
                "mean",
            ),

            daily_unweighted_mse=(
                "squared_total_variance_error",
                "mean",
            ),

            daily_mean_squared_iv_error=(
                "squared_iv_error",
                "mean",
            ),
        )
    )

    daily_metrics[
        "daily_iv_rmse_pp"
    ] = (
        100.0
        * np.sqrt(
            daily_metrics[
                "daily_mean_squared_iv_error"
            ]
        )
    )

    pooled_weighted_mse = (
        predictions[
            "weighted_squared_total_variance_error"
        ].mean()
    )

    pooled_unweighted_mse = (
        predictions[
            "squared_total_variance_error"
        ].mean()
    )

    pooled_iv_rmse_pp = (
        100.0
        * np.sqrt(
            predictions[
                "squared_iv_error"
            ].mean()
        )
    )

    rms_standardized_jump = np.sqrt(
        np.mean(
            standardized_changes**2
        )
    )

    mean_absolute_standardized_jump = (
        np.mean(
            np.abs(
                standardized_changes
            )
        )
    )

    maximum_absolute_standardized_jump = (
        np.max(
            np.abs(
                standardized_changes
            )
        )
    )

    lambda_summary_records.append(
        {
            "ridge_lambda": ridge_lambda,

            "lambda_label": (
                format_lambda(
                    ridge_lambda
                )
            ),

            # Fitting accuracy
            "pooled_weighted_mse": (
                pooled_weighted_mse
            ),

            "mean_daily_weighted_mse": (
                daily_metrics[
                    "daily_weighted_mse"
                ].mean()
            ),

            "pooled_unweighted_mse": (
                pooled_unweighted_mse
            ),

            "pooled_iv_rmse_pp": (
                pooled_iv_rmse_pp
            ),

            "mean_daily_iv_rmse_pp": (
                daily_metrics[
                    "daily_iv_rmse_pp"
                ].mean()
            ),

            "maximum_daily_iv_rmse_pp": (
                daily_metrics[
                    "daily_iv_rmse_pp"
                ].max()
            ),

            # Parameter smoothness
            "rms_standardized_jump": (
                rms_standardized_jump
            ),

            "mean_absolute_standardized_jump": (
                mean_absolute_standardized_jump
            ),

            "maximum_absolute_standardized_jump": (
                maximum_absolute_standardized_jump
            ),

            # Curve-quantity smoothness
            "mean_absolute_w_min_change": (
                parameters[
                    "w_min"
                ]
                .diff()
                .abs()
                .mean()
            ),

            "mean_absolute_minimum_location_change": (
                parameters[
                    "minimum_location_k"
                ]
                .diff()
                .abs()
                .mean()
            ),

            "mean_absolute_left_slope_change": (
                parameters[
                    "left_asymptotic_slope"
                ]
                .diff()
                .abs()
                .mean()
            ),

            # Objective diagnostics
            "mean_ridge_penalty": (
                parameters[
                    "ridge_penalty"
                ]
                .iloc[1:]
                .mean()
            ),

            "mean_total_objective": (
                parameters[
                    "total_objective"
                ].mean()
            ),

            # Numerical diagnostics
            "optimizer_successes": int(
                parameters[
                    "optimizer_success"
                ].sum()
            ),

            "dates_with_box_bound_hits": int(
                (
                    parameters[
                        "n_box_bound_hits"
                    ] > 0
                ).sum()
            ),

            "dates_with_active_constraints": int(
                (
                    parameters[
                        "n_active_constraints"
                    ] > 0
                ).sum()
            ),
        }
    )

lambda_summary = (
    pd.DataFrame(
        lambda_summary_records
    )
    .sort_values(
        "ridge_lambda"
    )
    .reset_index(drop=True)
)

display(
    lambda_summary
)

In [ ]:
#compare all lambdas with lambda = 0
lambda_zero_summary = (
    lambda_summary.loc[
        lambda_summary[
            "ridge_lambda"
        ] == 0.0
    ]
    .iloc[0]
)

lambda_summary[
    "iv_rmse_increase_percent"
] = (
    100.0
    * (
        lambda_summary[
            "pooled_iv_rmse_pp"
        ]
        / lambda_zero_summary[
            "pooled_iv_rmse_pp"
        ]
        - 1.0
    )
)

lambda_summary[
    "weighted_mse_increase_percent"
] = (
    100.0
    * (
        lambda_summary[
            "pooled_weighted_mse"
        ]
        / lambda_zero_summary[
            "pooled_weighted_mse"
        ]
        - 1.0
    )
)

lambda_summary[
    "rms_jump_reduction_percent"
] = (
    100.0
    * (
        1.0
        -
        lambda_summary[
            "rms_standardized_jump"
        ]
        / lambda_zero_summary[
            "rms_standardized_jump"
        ]
    )
)

lambda_summary[
    "mean_absolute_jump_reduction_percent"
] = (
    100.0
    * (
        1.0
        -
        lambda_summary[
            "mean_absolute_standardized_jump"
        ]
        / lambda_zero_summary[
            "mean_absolute_standardized_jump"
        ]
    )
)

lambda_summary[
    "maximum_jump_reduction_percent"
] = (
    100.0
    * (
        1.0
        -
        lambda_summary[
            "maximum_absolute_standardized_jump"
        ]
        / lambda_zero_summary[
            "maximum_absolute_standardized_jump"
        ]
    )
)

lambda_comparison_columns = [
    "ridge_lambda",
    "pooled_iv_rmse_pp",
    "iv_rmse_increase_percent",
    "pooled_weighted_mse",
    "weighted_mse_increase_percent",
    "rms_standardized_jump",
    "rms_jump_reduction_percent",
    "mean_absolute_standardized_jump",
    "mean_absolute_jump_reduction_percent",
    "maximum_absolute_standardized_jump",
    "maximum_jump_reduction_percent",
    "dates_with_box_bound_hits",
    "dates_with_active_constraints",
]

display(
    lambda_summary[
        lambda_comparison_columns
    ]
)

In [ ]:
#per parameter smoothing
parameter_smoothness_records = []

for ridge_lambda in RIDGE_LAMBDA_GRID:
    parameters = (
        regularized_results[
            ridge_lambda
        ]["parameters"]
    )

    for parameter in SVI_OPT_PARAMETER_NAMES:
        changes = (
            parameters[
                f"standardized_change_{parameter}"
            ]
            .iloc[1:]
            .to_numpy(
                dtype=float
            )
        )

        parameter_smoothness_records.append(
            {
                "ridge_lambda": (
                    ridge_lambda
                ),

                "lambda_label": (
                    format_lambda(
                        ridge_lambda
                    )
                ),

                "parameter": (
                    parameter
                ),

                "rms_standardized_jump": (
                    np.sqrt(
                        np.mean(
                            changes**2
                        )
                    )
                ),

                "mean_absolute_standardized_jump": (
                    np.mean(
                        np.abs(
                            changes
                        )
                    )
                ),

                "maximum_absolute_standardized_jump": (
                    np.max(
                        np.abs(
                            changes
                        )
                    )
                ),
            }
        )

parameter_smoothness = (
    pd.DataFrame(
        parameter_smoothness_records
    )
)

baseline_parameter_smoothness = (
    parameter_smoothness.loc[
        parameter_smoothness[
            "ridge_lambda"
        ] == 0.0,
        [
            "parameter",
            "rms_standardized_jump",
        ],
    ]
    .rename(
        columns={
            "rms_standardized_jump":
            "baseline_rms_standardized_jump"
        }
    )
)

parameter_smoothness = (
    parameter_smoothness
    .merge(
        baseline_parameter_smoothness,
        on="parameter",
        how="left",
        validate="many_to_one",
    )
)

parameter_smoothness[
    "rms_jump_reduction_percent"
] = (
    100.0
    * (
        1.0
        -
        parameter_smoothness[
            "rms_standardized_jump"
        ]
        / parameter_smoothness[
            "baseline_rms_standardized_jump"
        ]
    )
)

parameter_jump_pivot = (
    parameter_smoothness
    .pivot(
        index="ridge_lambda",
        columns="parameter",
        values="rms_standardized_jump",
    )
    .loc[
        RIDGE_LAMBDA_GRID,
        SVI_OPT_PARAMETER_NAMES,
    ]
)

display(
    parameter_jump_pivot
)

In [ ]:
#identify plausible candidates
MAX_ACCEPTABLE_IV_RMSE_INCREASE_PERCENT = 5.0

lambda_summary[
    "within_fit_tolerance"
] = (
    lambda_summary[
        "iv_rmse_increase_percent"
    ]
    <= MAX_ACCEPTABLE_IV_RMSE_INCREASE_PERCENT
)

plausible_lambda_candidates = (
    lambda_summary.loc[
        lambda_summary[
            "within_fit_tolerance"
        ]
    ]
    .sort_values(
        [
            "rms_jump_reduction_percent",
            "iv_rmse_increase_percent",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    plausible_lambda_candidates[
        [
            "ridge_lambda",
            "pooled_iv_rmse_pp",
            "iv_rmse_increase_percent",
            "pooled_weighted_mse",
            "weighted_mse_increase_percent",
            "rms_standardized_jump",
            "rms_jump_reduction_percent",
            "maximum_absolute_standardized_jump",
            "maximum_jump_reduction_percent",
        ]
    ]
)

In [ ]:
#one standardized parameter-path chart for each lambda.
baseline_optimization_means = (
    baseline_parameters[
        SVI_OPT_PARAMETER_NAMES
    ]
    .mean()
)

for ridge_lambda in RIDGE_LAMBDA_GRID:
    parameters = (
        regularized_results[
            ridge_lambda
        ]["parameters"]
    )

    plt.figure(
        figsize=(11, 6)
    )

    for parameter_index, parameter in enumerate(
        SVI_OPT_PARAMETER_NAMES
    ):
        standardized_level = (
            (
                parameters[
                    parameter
                ]
                - baseline_optimization_means[
                    parameter
                ]
            )
            / ridge_scale_array[
                parameter_index
            ]
        )

        plt.plot(
            parameters[
                "price_date"
            ],
            standardized_level,
            marker="o",
            markersize=2,
            label=parameter,
        )

    plt.axhline(
        0,
        linestyle="--",
        alpha=0.6,
    )

    plt.xlabel(
        "Observation date"
    )

    plt.ylabel(
        "Parameter level in baseline SDs"
    )

    plt.title(
        "Regularized SVI Parameter Paths: "
        f"lambda = "
        f"{format_lambda(ridge_lambda)}"
    )

    plt.xticks(
        rotation=45
    )

    plt.legend(
        ncol=5
    )

    plt.tight_layout()
    plt.show()

In [ ]:
#fit vs smoothness
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    lambda_summary[
        "rms_jump_reduction_percent"
    ],
    lambda_summary[
        "iv_rmse_increase_percent"
    ],
    marker="o",
)

for _, row in lambda_summary.iterrows():
    plt.annotate(
        row[
            "lambda_label"
        ],
        (
            row[
                "rms_jump_reduction_percent"
            ],
            row[
                "iv_rmse_increase_percent"
            ],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.axhline(
    MAX_ACCEPTABLE_IV_RMSE_INCREASE_PERCENT,
    linestyle="--",
    alpha=0.7,
)

plt.xlabel(
    "Reduction in RMS standardized "
    "parameter jumps (%)"
)

plt.ylabel(
    "Increase in pooled IV RMSE (%)"
)

plt.title(
    "Temporal Ridge Fit–Smoothness Tradeoff"
)

plt.tight_layout()
plt.show()

In [ ]:
#save all lambda results
regularized_all_predictions.to_csv(
    OUTPUT_DIR
    / "regularized_svi_all_lambda_predictions.csv",
    index=False,
)

regularized_all_parameters.to_csv(
    OUTPUT_DIR
    / "regularized_svi_all_lambda_parameters.csv",
    index=False,
)

lambda_summary.to_csv(
    OUTPUT_DIR
    / "regularized_svi_lambda_summary.csv",
    index=False,
)

parameter_smoothness.to_csv(
    OUTPUT_DIR
    / "regularized_svi_parameter_smoothness.csv",
    index=False,
)

plausible_lambda_candidates.to_csv(
    OUTPUT_DIR
    / "regularized_svi_plausible_lambda_candidates.csv",
    index=False,
)

for ridge_lambda in RIDGE_LAMBDA_GRID:
    lambda_filename_label = (
        format_lambda(
            ridge_lambda
        )
        .replace("-", "minus")
    )

    regularized_results[
        ridge_lambda
    ]["predictions"].to_csv(
        OUTPUT_DIR
        / (
            "regularized_svi_predictions_"
            f"lambda_{lambda_filename_label}.csv"
        ),
        index=False,
    )

    regularized_results[
        ridge_lambda
    ]["parameters"].to_csv(
        OUTPUT_DIR
        / (
            "regularized_svi_parameters_"
            f"lambda_{lambda_filename_label}.csv"
        ),
        index=False,
    )

print(
    "Regularized lambda-grid "
    "results saved."
)

The final regularized model achieves the desired stability/fit tradeoff.

lambda = 1e-6 is probably the best, since RMS standardized jumps fall by 80%
worst one-day jump is 8.31 to 1.02 SD
IV RMSE only rises from 0.7542 to 0.7574
(Absolute deterioration only 0.003208 volatility percentage points) - the fitting cost is negligible
The parameter paths still have meaningful movement. They are smooth but not flat.

The tradeoff curve begins bending upward around 1e-6 to 3e-6. (IV-RMSE deterioration more than doubles)
3e-7 is defensible, but 1e-6 improves jump reduction almost 10% and reduces maximum jump to almost exactly one baseline SD while only sacrificing 0.021 RMSE volatility points

At lambda = 1e-6, the RMSE standardized daily changes are
w_min       0.310
b           0.065
rho         0.098
m           0.177
svi_sigma   0.235

The mean absolute change for minimum_location_k didn't decline, but that's possible since it's determined jointly by several parameters.

Moving on to the outputs now.

In [ ]:
from pathlib import Path
import json

current_directory = Path.cwd().resolve()

if current_directory.name == "notebooks":
    PROJECT_ROOT = current_directory.parent

elif (
    current_directory
    / "notebooks"
).exists():
    PROJECT_ROOT = current_directory

else:
    raise RuntimeError(
        "Could not identify the project root. "
        "Run this notebook from either the "
        "project root or the notebooks folder."
    )

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
)

TABLE_DIR = (
    OUTPUT_DIR
    / "tables"
)

FIGURE_DIR = (
    OUTPUT_DIR
    / "figures"
)

LAMBDA_FIGURE_DIR = (
    FIGURE_DIR
    / "lambda_parameter_paths"
)

for directory in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    LAMBDA_FIGURE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

FINAL_RIDGE_LAMBDA = 1e-6

assert (
    FINAL_RIDGE_LAMBDA
    in regularized_results
)

final_predictions = (
    regularized_results[
        FINAL_RIDGE_LAMBDA
    ]["predictions"]
    .copy()
)

final_parameters = (
    regularized_results[
        FINAL_RIDGE_LAMBDA
    ]["parameters"]
    .copy()
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Outputs will be saved to:",
    OUTPUT_DIR,
)

print(
    "Final lambda:",
    FINAL_RIDGE_LAMBDA,
)

In [ ]:
#core result tables
def make_daily_fit_metrics(
    predictions,
    model_label,
):
    daily_metrics = (
        predictions
        .groupby(
            [
                "price_date",
                "option_expiry",
            ],
            as_index=False,
        )
        .agg(
            n_options=(
                "iv_error_pp",
                "size",
            ),

            weighted_mse_total_variance=(
                "weighted_squared_total_variance_error",
                "mean",
            ),

            unweighted_mse_total_variance=(
                "squared_total_variance_error",
                "mean",
            ),

            mean_squared_iv_error=(
                "squared_iv_error",
                "mean",
            ),

            mean_iv_error_pp=(
                "iv_error_pp",
                "mean",
            ),

            mae_iv_pp=(
                "iv_error_pp",
                lambda values:
                values.abs().mean(),
            ),

            maximum_absolute_iv_error_pp=(
                "iv_error_pp",
                lambda values:
                values.abs().max(),
            ),
        )
    )

    daily_metrics[
        "rmse_total_variance"
    ] = np.sqrt(
        daily_metrics[
            "unweighted_mse_total_variance"
        ]
    )

    daily_metrics[
        "rmse_iv_percentage_points"
    ] = (
        100.0
        * np.sqrt(
            daily_metrics[
                "mean_squared_iv_error"
            ]
        )
    )

    daily_metrics.insert(
        0,
        "model",
        model_label,
    )

    return daily_metrics


baseline_daily_export = (
    make_daily_fit_metrics(
        baseline_predictions,
        "Independent baseline",
    )
)

final_daily_metrics = (
    make_daily_fit_metrics(
        final_predictions,
        (
            "Final ridge "
            f"(lambda={FINAL_RIDGE_LAMBDA:.0e})"
        ),
    )
)

daily_fit_comparison = (
    pd.concat(
        [
            baseline_daily_export,
            final_daily_metrics,
        ],
        ignore_index=True,
    )
)

In [ ]:
#overall comparison
def make_overall_fit_metrics(
    predictions,
    daily_metrics,
    model_label,
):
    return {
        "model": model_label,

        "n_observations": (
            len(predictions)
        ),

        "n_dates": (
            predictions[
                "price_date"
            ].nunique()
        ),

        "weighted_mse_total_variance": (
            predictions[
                "weighted_squared_total_variance_error"
            ].mean()
        ),

        "unweighted_mse_total_variance": (
            predictions[
                "squared_total_variance_error"
            ].mean()
        ),

        "rmse_total_variance": np.sqrt(
            predictions[
                "squared_total_variance_error"
            ].mean()
        ),

        "pooled_iv_rmse_pp": (
            100.0
            * np.sqrt(
                predictions[
                    "squared_iv_error"
                ].mean()
            )
        ),

        "mean_daily_iv_rmse_pp": (
            daily_metrics[
                "rmse_iv_percentage_points"
            ].mean()
        ),

        "maximum_daily_iv_rmse_pp": (
            daily_metrics[
                "rmse_iv_percentage_points"
            ].max()
        ),

        "mean_iv_error_pp": (
            predictions[
                "iv_error_pp"
            ].mean()
        ),

        "mae_iv_pp": (
            predictions[
                "iv_error_pp"
            ].abs().mean()
        ),
    }


overall_model_comparison = (
    pd.DataFrame(
        [
            make_overall_fit_metrics(
                baseline_predictions,
                baseline_daily_export,
                "Independent baseline",
            ),

            make_overall_fit_metrics(
                final_predictions,
                final_daily_metrics,
                (
                    "Final ridge "
                    f"(lambda="
                    f"{FINAL_RIDGE_LAMBDA:.0e})"
                ),
            ),
        ]
    )
)

display(
    overall_model_comparison
)

#saving
model_data.to_csv(
    TABLE_DIR
    / "model_ready_option_quotes.csv",
    index=False,
)

baseline_predictions.to_csv(
    TABLE_DIR
    / "baseline_predictions.csv",
    index=False,
)

baseline_parameters.to_csv(
    TABLE_DIR
    / "baseline_parameters.csv",
    index=False,
)

baseline_daily_export.to_csv(
    TABLE_DIR
    / "baseline_daily_fit_metrics.csv",
    index=False,
)

final_predictions.to_csv(
    TABLE_DIR
    / "final_ridge_predictions.csv",
    index=False,
)

final_parameters.to_csv(
    TABLE_DIR
    / "final_ridge_parameters.csv",
    index=False,
)

final_daily_metrics.to_csv(
    TABLE_DIR
    / "final_ridge_daily_fit_metrics.csv",
    index=False,
)

daily_fit_comparison.to_csv(
    TABLE_DIR
    / "baseline_vs_final_daily_metrics.csv",
    index=False,
)

overall_model_comparison.to_csv(
    TABLE_DIR
    / "baseline_vs_final_overall_metrics.csv",
    index=False,
)

lambda_summary.to_csv(
    TABLE_DIR
    / "lambda_summary.csv",
    index=False,
)

parameter_smoothness.to_csv(
    TABLE_DIR
    / "lambda_parameter_smoothness.csv",
    index=False,
)

ridge_parameter_scales.to_csv(
    TABLE_DIR
    / "ridge_parameter_scales.csv",
)

lambda_summary.loc[
    lambda_summary[
        "ridge_lambda"
    ] == FINAL_RIDGE_LAMBDA
].to_csv(
    TABLE_DIR
    / "selected_lambda_summary.csv",
    index=False,
)

regularized_all_parameters.to_csv(
    TABLE_DIR
    / "all_lambda_parameters.csv",
    index=False,
)

regularized_all_predictions.to_csv(
    TABLE_DIR
    / "all_lambda_predictions.csv",
    index=False,
)

print(
    "Core tables saved."
)

In [ ]:
#model metadata
final_model_metadata = {
    "model": "Raw SVI",

    "expiry_structure": (
        "One fixed option expiry observed "
        "across multiple dates"
    ),

    "final_ridge_lambda": (
        FINAL_RIDGE_LAMBDA
    ),

    "ridge_parameters": (
        list(
            SVI_OPT_PARAMETER_NAMES
        )
    ),

    "ridge_scales": {
        parameter: float(
            ridge_parameter_scales.loc[
                parameter,
                "ridge_scale",
            ]
        )
        for parameter
        in SVI_OPT_PARAMETER_NAMES
    },

    "objective": (
        "Weighted total-variance MSE plus "
        "lambda times mean squared "
        "standardized change from the "
        "previous date"
    ),

    "n_dates": int(
        final_parameters[
            "price_date"
        ].nunique()
    ),

    "n_observations": int(
        len(
            final_predictions
        )
    ),

    "final_pooled_iv_rmse_pp": float(
        overall_model_comparison.loc[
            overall_model_comparison[
                "model"
            ].str.startswith(
                "Final ridge"
            ),
            "pooled_iv_rmse_pp",
        ].iloc[0]
    ),
}

with open(
    OUTPUT_DIR
    / "final_model_metadata.json",
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        final_model_metadata,
        metadata_file,
        indent=2,
    )

print(
    "Final-model metadata saved."
)

In [ ]:
#save the lambda tradeoff plot
fig, axis = plt.subplots(
    figsize=(8, 6)
)

axis.plot(
    lambda_summary[
        "rms_jump_reduction_percent"
    ],
    lambda_summary[
        "iv_rmse_increase_percent"
    ],
    marker="o",
)

for _, row in lambda_summary.iterrows():
    axis.annotate(
        row[
            "lambda_label"
        ],
        (
            row[
                "rms_jump_reduction_percent"
            ],
            row[
                "iv_rmse_increase_percent"
            ],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

selected_lambda_row = (
    lambda_summary.loc[
        lambda_summary[
            "ridge_lambda"
        ] == FINAL_RIDGE_LAMBDA
    ]
    .iloc[0]
)

axis.scatter(
    selected_lambda_row[
        "rms_jump_reduction_percent"
    ],
    selected_lambda_row[
        "iv_rmse_increase_percent"
    ],
    marker="*",
    s=180,
    label=(
        "Selected lambda "
        f"= {FINAL_RIDGE_LAMBDA:.0e}"
    ),
)

axis.set_xlabel(
    "Reduction in RMS standardized "
    "parameter jumps (%)"
)

axis.set_ylabel(
    "Increase in pooled IV RMSE (%)"
)

axis.set_title(
    "Temporal Ridge Fit–Smoothness Tradeoff"
)

axis.legend()

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "lambda_fit_smoothness_tradeoff.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR
    / "lambda_fit_smoothness_tradeoff.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# Extended lambda stress test:
# includes all original lambdas plus larger values

required_names = [
    "run_regularized_svi_path",
    "regularized_results",
    "SVI_OPT_PARAMETER_NAMES",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "This cell must be run in the original notebook after "
        "the regularized-model functions and paths have been run. "
        f"Missing: {missing_names}"
    )


ORIGINAL_LAMBDAS = [
    0.0,
    1e-8,
    3e-8,
    1e-7,
    3e-7,
    1e-6,
    3e-6,
    1e-5,
    3e-5,
    1e-4,
]

STRESS_LAMBDAS = [
    3e-4,
    1e-3,
    3e-3,
    1e-2,
]

ALL_LAMBDAS = sorted(
    set(
        ORIGINAL_LAMBDAS
        + STRESS_LAMBDAS
    )
)

# Copy the existing results so nothing earlier is overwritten.
extended_regularized_results = dict(
    regularized_results
)

# Only calculate lambda paths that do not already exist.
for ridge_lambda in ALL_LAMBDAS:
    if ridge_lambda not in extended_regularized_results:
        print(
            f"Running lambda = {ridge_lambda:.0e}"
        )

        extended_regularized_results[
            ridge_lambda
        ] = run_regularized_svi_path(
            ridge_lambda
        )


change_columns = [
    f"standardized_change_{parameter}"
    for parameter in SVI_OPT_PARAMETER_NAMES
]

summary_records = []

for ridge_lambda in ALL_LAMBDAS:
    predictions = (
        extended_regularized_results[
            ridge_lambda
        ]["predictions"]
    )

    parameters = (
        extended_regularized_results[
            ridge_lambda
        ]["parameters"]
    )

    iv_rmse_pp = (
        100.0
        * np.sqrt(
            predictions[
                "squared_iv_error"
            ].mean()
        )
    )

    weighted_mse = (
        predictions[
            "weighted_squared_total_variance_error"
        ].mean()
    )

    standardized_changes = (
        parameters[
            change_columns
        ]
        .iloc[1:]
        .to_numpy(
            dtype=float
        )
    )

    rms_jump = np.sqrt(
        np.mean(
            standardized_changes**2
        )
    )

    maximum_jump = np.max(
        np.abs(
            standardized_changes
        )
    )

    summary_records.append(
        {
            "ridge_lambda": ridge_lambda,
            "lambda_label": (
                "0"
                if ridge_lambda == 0
                else f"{ridge_lambda:.0e}"
            ),
            "pooled_iv_rmse_pp": iv_rmse_pp,
            "pooled_weighted_mse": weighted_mse,
            "rms_standardized_jump": rms_jump,
            "maximum_standardized_jump": maximum_jump,
            "optimizer_successes": int(
                parameters[
                    "optimizer_success"
                ].sum()
            ),
        }
    )


extended_lambda_summary = (
    pd.DataFrame(
        summary_records
    )
    .sort_values(
        "ridge_lambda"
    )
    .reset_index(drop=True)
)

baseline_row = (
    extended_lambda_summary.loc[
        extended_lambda_summary[
            "ridge_lambda"
        ] == 0.0
    ]
    .iloc[0]
)

extended_lambda_summary[
    "absolute_iv_rmse_increase_pp"
] = (
    extended_lambda_summary[
        "pooled_iv_rmse_pp"
    ]
    -
    baseline_row[
        "pooled_iv_rmse_pp"
    ]
)

extended_lambda_summary[
    "relative_iv_rmse_increase_percent"
] = (
    100.0
    * (
        extended_lambda_summary[
            "pooled_iv_rmse_pp"
        ]
        / baseline_row[
            "pooled_iv_rmse_pp"
        ]
        - 1.0
    )
)

extended_lambda_summary[
    "rms_jump_reduction_percent"
] = (
    100.0
    * (
        1.0
        -
        extended_lambda_summary[
            "rms_standardized_jump"
        ]
        / baseline_row[
            "rms_standardized_jump"
        ]
    )
)

extended_lambda_summary[
    "maximum_jump_reduction_percent"
] = (
    100.0
    * (
        1.0
        -
        extended_lambda_summary[
            "maximum_standardized_jump"
        ]
        / baseline_row[
            "maximum_standardized_jump"
        ]
    )
)

display(
    extended_lambda_summary[
        [
            "ridge_lambda",
            "pooled_iv_rmse_pp",
            "absolute_iv_rmse_increase_pp",
            "relative_iv_rmse_increase_percent",
            "rms_standardized_jump",
            "rms_jump_reduction_percent",
            "maximum_standardized_jump",
            "maximum_jump_reduction_percent",
            "optimizer_successes",
        ]
    ]
)


# Combined graph for all original and stress-test lambdas

lambda_positions = np.arange(
    len(
        extended_lambda_summary
    )
)

plt.figure(
    figsize=(11, 6)
)

plt.plot(
    lambda_positions,
    extended_lambda_summary[
        "absolute_iv_rmse_increase_pp"
    ],
    marker="o",
)

selected_position = (
    extended_lambda_summary.index[
        extended_lambda_summary[
            "ridge_lambda"
        ] == 1e-6
    ][0]
)

plt.axvline(
    selected_position,
    linestyle="--",
    alpha=0.7,
    label="Selected lambda = 1e-6",
)

plt.axhline(
    0,
    linestyle="--",
    alpha=0.5,
)

plt.xticks(
    lambda_positions,
    extended_lambda_summary[
        "lambda_label"
    ],
    rotation=45,
)

plt.xlabel(
    "Ridge lambda"
)

plt.ylabel(
    "Absolute increase in pooled IV RMSE "
    "(volatility percentage points)"
)

plt.title(
    "Extended Temporal-Ridge Stress Test"
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#save a parameter cell path chart for each lambda
baseline_parameter_means = (
    baseline_parameters[
        SVI_OPT_PARAMETER_NAMES
    ]
    .mean()
)


def make_safe_lambda_label(
    ridge_lambda,
):
    return (
        format_lambda(
            ridge_lambda
        )
        .replace("-", "minus")
        .replace("+", "plus")
    )


for ridge_lambda in RIDGE_LAMBDA_GRID:
    parameters = (
        regularized_results[
            ridge_lambda
        ]["parameters"]
    )

    fig, axis = plt.subplots(
        figsize=(11, 6)
    )

    for parameter_index, parameter in enumerate(
        SVI_OPT_PARAMETER_NAMES
    ):
        standardized_level = (
            (
                parameters[
                    parameter
                ]
                - baseline_parameter_means[
                    parameter
                ]
            )
            / ridge_scale_array[
                parameter_index
            ]
        )

        axis.plot(
            parameters[
                "price_date"
            ],
            standardized_level,
            marker="o",
            markersize=2,
            label=parameter,
        )

    axis.axhline(
        0,
        linestyle="--",
        alpha=0.6,
    )

    axis.set_xlabel(
        "Observation date"
    )

    axis.set_ylabel(
        "Parameter level in baseline SDs"
    )

    axis.set_title(
        "Regularized SVI Parameter Paths: "
        f"lambda = "
        f"{format_lambda(ridge_lambda)}"
    )

    axis.tick_params(
        axis="x",
        rotation=45,
    )

    axis.legend(
        ncol=5
    )

    fig.tight_layout()

    lambda_file_label = (
        make_safe_lambda_label(
            ridge_lambda
        )
    )

    fig.savefig(
        LAMBDA_FIGURE_DIR
        / (
            "parameter_paths_"
            f"lambda_{lambda_file_label}.png"
        ),
        dpi=220,
        bbox_inches="tight",
    )

    plt.close(fig)

print(
    "All lambda parameter-path "
    "figures saved."
)

In [ ]:
#save baseline-vs-final parameter charts
for parameter in SVI_OPT_PARAMETER_NAMES:
    fig, axis = plt.subplots(
        figsize=(10, 5)
    )

    axis.plot(
        baseline_parameters[
            "price_date"
        ],
        baseline_parameters[
            parameter
        ],
        marker="o",
        markersize=2,
        label="Independent baseline",
    )

    axis.plot(
        final_parameters[
            "price_date"
        ],
        final_parameters[
            parameter
        ],
        marker="o",
        markersize=2,
        label=(
            "Final ridge "
            f"(lambda={FINAL_RIDGE_LAMBDA:.0e})"
        ),
    )

    axis.set_xlabel(
        "Observation date"
    )

    axis.set_ylabel(
        parameter
    )

    axis.set_title(
        "Baseline versus Final SVI "
        f"Parameter: {parameter}"
    )

    axis.tick_params(
        axis="x",
        rotation=45,
    )

    axis.legend()

    fig.tight_layout()

    fig.savefig(
        FIGURE_DIR
        / (
            "baseline_vs_final_"
            f"parameter_{parameter}.png"
        ),
        dpi=220,
        bbox_inches="tight",
    )

    plt.close(fig)

print(
    "Baseline-versus-final parameter "
    "charts saved."
)

In [ ]:
#daily IV-RMSE comparison
baseline_daily_plot = (
    baseline_daily_export
    .sort_values(
        "price_date"
    )
)

final_daily_plot = (
    final_daily_metrics
    .sort_values(
        "price_date"
    )
)

fig, axis = plt.subplots(
    figsize=(10, 5)
)

axis.plot(
    baseline_daily_plot[
        "price_date"
    ],
    baseline_daily_plot[
        "rmse_iv_percentage_points"
    ],
    marker="o",
    markersize=2,
    label="Independent baseline",
)

axis.plot(
    final_daily_plot[
        "price_date"
    ],
    final_daily_plot[
        "rmse_iv_percentage_points"
    ],
    marker="o",
    markersize=2,
    label=(
        "Final ridge "
        f"(lambda={FINAL_RIDGE_LAMBDA:.0e})"
    ),
)

axis.set_xlabel(
    "Observation date"
)

axis.set_ylabel(
    "IV RMSE (percentage points)"
)

axis.set_title(
    "Daily SVI Calibration Error"
)

axis.tick_params(
    axis="x",
    rotation=45,
)

axis.legend()

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "baseline_vs_final_daily_iv_rmse.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR
    / "baseline_vs_final_daily_iv_rmse.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
#save all fits (PDF)
from matplotlib.backends.backend_pdf import (
    PdfPages
)

all_fit_pdf_path = (
    FIGURE_DIR
    / "all_dates_market_baseline_final_fits.pdf"
)

with PdfPages(
    all_fit_pdf_path
) as pdf:
    for selected_date in available_dates:
        baseline_date = (
            baseline_predictions.loc[
                baseline_predictions[
                    "price_date"
                ] == selected_date
            ]
            .sort_values(
                "log_moneyness"
            )
            .reset_index(drop=True)
        )

        final_date = (
            final_predictions.loc[
                final_predictions[
                    "price_date"
                ] == selected_date
            ]
            .sort_values(
                "log_moneyness"
            )
            .reset_index(drop=True)
        )

        assert np.allclose(
            baseline_date[
                "log_moneyness"
            ],
            final_date[
                "log_moneyness"
            ],
        )

        fig, axes = plt.subplots(
            1,
            2,
            figsize=(13, 5),
        )

        # Implied-volatility panel
        axes[0].scatter(
            baseline_date[
                "log_moneyness"
            ],
            100.0
            * baseline_date[
                "iv_decimal"
            ],
            s=12,
            label="Market IV",
        )

        axes[0].plot(
            baseline_date[
                "log_moneyness"
            ],
            100.0
            * baseline_date[
                "predicted_iv"
            ],
            label="Independent SVI",
        )

        axes[0].plot(
            final_date[
                "log_moneyness"
            ],
            100.0
            * final_date[
                "predicted_iv"
            ],
            label=(
                "Final ridge SVI"
            ),
        )

        axes[0].set_xlabel(
            "Log-moneyness"
        )

        axes[0].set_ylabel(
            "Implied volatility (%)"
        )

        axes[0].set_title(
            "Implied-Volatility Fit"
        )

        axes[0].legend()

        # Total-variance panel
        axes[1].scatter(
            baseline_date[
                "log_moneyness"
            ],
            baseline_date[
                "market_total_variance"
            ],
            s=12,
            label="Market total variance",
        )

        axes[1].plot(
            baseline_date[
                "log_moneyness"
            ],
            baseline_date[
                "predicted_total_variance"
            ],
            label="Independent SVI",
        )

        axes[1].plot(
            final_date[
                "log_moneyness"
            ],
            final_date[
                "predicted_total_variance"
            ],
            label="Final ridge SVI",
        )

        axes[1].set_xlabel(
            "Log-moneyness"
        )

        axes[1].set_ylabel(
            "Total implied variance"
        )

        axes[1].set_title(
            "Total-Variance Fit"
        )

        axes[1].legend()

        fig.suptitle(
            "Fixed-Expiry SVI Calibration: "
            f"{pd.Timestamp(selected_date).date()}"
        )

        fig.tight_layout()

        pdf.savefig(
            fig,
            bbox_inches="tight",
        )

        plt.close(fig)

print(
    "Saved:",
    all_fit_pdf_path,
)

In [ ]:
#save final residual heatmap
residual_heatmap_data = (
    final_predictions.copy()
)

N_MONEYNESS_BINS = 40

moneyness_bin_edges = np.linspace(
    residual_heatmap_data[
        "log_moneyness"
    ].min(),
    residual_heatmap_data[
        "log_moneyness"
    ].max(),
    N_MONEYNESS_BINS + 1,
)

residual_heatmap_data[
    "moneyness_bin"
] = pd.cut(
    residual_heatmap_data[
        "log_moneyness"
    ],
    bins=moneyness_bin_edges,
    include_lowest=True,
)

residual_heatmap_summary = (
    residual_heatmap_data
    .groupby(
        [
            "price_date",
            "moneyness_bin",
        ],
        observed=False,
        as_index=False,
    )
    .agg(
        mean_iv_error_pp=(
            "iv_error_pp",
            "mean",
        )
    )
)

residual_heatmap_summary[
    "moneyness_bin_midpoint"
] = (
    residual_heatmap_summary[
        "moneyness_bin"
    ]
    .apply(
        lambda interval:
        interval.mid
        if pd.notna(interval)
        else np.nan
    )
    .astype(float)
)

residual_heatmap_pivot = (
    residual_heatmap_summary
    .pivot(
        index="price_date",
        columns="moneyness_bin_midpoint",
        values="mean_iv_error_pp",
    )
    .sort_index()
)

fig, axis = plt.subplots(
    figsize=(12, 7)
)

image = axis.imshow(
    residual_heatmap_pivot.to_numpy(),
    aspect="auto",
    origin="lower",
    interpolation="nearest",
)

colorbar = fig.colorbar(
    image,
    ax=axis,
)

colorbar.set_label(
    "Mean IV residual "
    "(percentage points)"
)

x_tick_positions = np.linspace(
    0,
    residual_heatmap_pivot.shape[1] - 1,
    8,
).astype(int)

axis.set_xticks(
    x_tick_positions
)

axis.set_xticklabels(
    [
        f"{residual_heatmap_pivot.columns[position]:.2f}"
        for position
        in x_tick_positions
    ]
)

y_tick_positions = np.linspace(
    0,
    residual_heatmap_pivot.shape[0] - 1,
    7,
).astype(int)

axis.set_yticks(
    y_tick_positions
)

axis.set_yticklabels(
    [
        pd.Timestamp(
            residual_heatmap_pivot.index[
                position
            ]
        ).strftime("%Y-%m-%d")
        for position
        in y_tick_positions
    ]
)

axis.set_xlabel(
    "Log-moneyness bin"
)

axis.set_ylabel(
    "Observation date"
)

axis.set_title(
    "Final Ridge-SVI Residual Heatmap"
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "final_residual_heatmap.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR
    / "final_residual_heatmap.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
REPRESENTATIVE_FIGURE_DIR = (
    FIGURE_DIR
    / "representative_smiles"
)

REPRESENTATIVE_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for selected_date in representative_dates:
    baseline_date = (
        baseline_predictions.loc[
            baseline_predictions[
                "price_date"
            ] == selected_date
        ]
        .sort_values(
            "log_moneyness"
        )
        .reset_index(drop=True)
    )

    final_date = (
        final_predictions.loc[
            final_predictions[
                "price_date"
            ] == selected_date
        ]
        .sort_values(
            "log_moneyness"
        )
        .reset_index(drop=True)
    )

    assert len(baseline_date) > 0

    assert np.allclose(
        baseline_date[
            "log_moneyness"
        ],
        final_date[
            "log_moneyness"
        ],
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5),
    )

    # Implied-volatility smile
    axes[0].scatter(
        baseline_date[
            "log_moneyness"
        ],
        100.0
        * baseline_date[
            "iv_decimal"
        ],
        s=16,
        label="Market IV",
    )

    axes[0].plot(
        baseline_date[
            "log_moneyness"
        ],
        100.0
        * baseline_date[
            "predicted_iv"
        ],
        label="Independent SVI",
    )

    axes[0].plot(
        final_date[
            "log_moneyness"
        ],
        100.0
        * final_date[
            "predicted_iv"
        ],
        label=(
            "Final ridge SVI "
            f"(lambda={FINAL_RIDGE_LAMBDA:.0e})"
        ),
    )

    axes[0].set_xlabel(
        "Log-moneyness"
    )

    axes[0].set_ylabel(
        "Implied volatility (%)"
    )

    axes[0].set_title(
        "Implied-Volatility Smile"
    )

    axes[0].legend()

    # Total-variance smile
    axes[1].scatter(
        baseline_date[
            "log_moneyness"
        ],
        baseline_date[
            "market_total_variance"
        ],
        s=16,
        label="Market total variance",
    )

    axes[1].plot(
        baseline_date[
            "log_moneyness"
        ],
        baseline_date[
            "predicted_total_variance"
        ],
        label="Independent SVI",
    )

    axes[1].plot(
        final_date[
            "log_moneyness"
        ],
        final_date[
            "predicted_total_variance"
        ],
        label="Final ridge SVI",
    )

    axes[1].set_xlabel(
        "Log-moneyness"
    )

    axes[1].set_ylabel(
        "Total implied variance"
    )

    axes[1].set_title(
        "Total-Variance Smile"
    )

    axes[1].legend()

    date_label = (
        pd.Timestamp(
            selected_date
        ).strftime("%Y-%m-%d")
    )

    fig.suptitle(
        "Fixed-Expiry SVI Calibration: "
        f"{date_label}"
    )

    fig.tight_layout()

    fig.savefig(
        REPRESENTATIVE_FIGURE_DIR
        / (
            "representative_smile_"
            f"{date_label}.png"
        ),
        dpi=250,
        bbox_inches="tight",
    )

    fig.savefig(
        REPRESENTATIVE_FIGURE_DIR
        / (
            "representative_smile_"
            f"{date_label}.pdf"
        ),
        bbox_inches="tight",
    )

    plt.close(fig)

print(
    "Representative smile figures saved."
)

In [ ]:
#five worst final model dates
WORST_DATE_FIGURE_DIR = (
    FIGURE_DIR
    / "worst_date_fits"
)

WORST_DATE_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

final_worst_dates = (
    final_daily_metrics
    .nlargest(
        5,
        "rmse_iv_percentage_points",
    )
    .copy()
)

final_worst_dates.to_csv(
    TABLE_DIR
    / "final_model_worst_fitting_dates.csv",
    index=False,
)

display(
    final_worst_dates[
        [
            "price_date",
            "n_options",
            "weighted_mse_total_variance",
            "rmse_iv_percentage_points",
            "maximum_absolute_iv_error_pp",
        ]
    ]
)

for selected_date in final_worst_dates[
    "price_date"
]:
    baseline_date = (
        baseline_predictions.loc[
            baseline_predictions[
                "price_date"
            ] == selected_date
        ]
        .sort_values(
            "log_moneyness"
        )
    )

    final_date = (
        final_predictions.loc[
            final_predictions[
                "price_date"
            ] == selected_date
        ]
        .sort_values(
            "log_moneyness"
        )
    )

    final_daily_row = (
        final_worst_dates.loc[
            final_worst_dates[
                "price_date"
            ] == selected_date
        ]
        .iloc[0]
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5),
    )

    axes[0].scatter(
        baseline_date[
            "log_moneyness"
        ],
        100.0
        * baseline_date[
            "iv_decimal"
        ],
        s=15,
        label="Market IV",
    )

    axes[0].plot(
        baseline_date[
            "log_moneyness"
        ],
        100.0
        * baseline_date[
            "predicted_iv"
        ],
        label="Independent SVI",
    )

    axes[0].plot(
        final_date[
            "log_moneyness"
        ],
        100.0
        * final_date[
            "predicted_iv"
        ],
        label="Final ridge SVI",
    )

    axes[0].set_xlabel(
        "Log-moneyness"
    )

    axes[0].set_ylabel(
        "Implied volatility (%)"
    )

    axes[0].set_title(
        "Implied-Volatility Fit"
    )

    axes[0].legend()

    axes[1].scatter(
        final_date[
            "log_moneyness"
        ],
        final_date[
            "iv_error_pp"
        ],
        s=15,
    )

    axes[1].axhline(
        0,
        linestyle="--",
    )

    axes[1].set_xlabel(
        "Log-moneyness"
    )

    axes[1].set_ylabel(
        "IV residual "
        "(percentage points)"
    )

    axes[1].set_title(
        "Final-Model Residuals"
    )

    date_label = (
        pd.Timestamp(
            selected_date
        ).strftime("%Y-%m-%d")
    )

    fig.suptitle(
        f"{date_label}: "
        "Final IV RMSE = "
        f"{final_daily_row['rmse_iv_percentage_points']:.3f} pp"
    )

    fig.tight_layout()

    fig.savefig(
        WORST_DATE_FIGURE_DIR
        / (
            "worst_date_fit_"
            f"{date_label}.png"
        ),
        dpi=250,
        bbox_inches="tight",
    )

    plt.close(fig)

print(
    "Worst-date figures saved."
)

In [ ]:
#raw and derived parameter comparisons
PARAMETER_COMPARISON_DIR = (
    FIGURE_DIR
    / "parameter_comparisons"
)

PARAMETER_COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RAW_PARAMETER_NAMES = [
    "a",
    "b",
    "rho",
    "m",
    "svi_sigma",
]

DERIVED_QUANTITY_NAMES = [
    "w_min",
    "minimum_location_k",
    "left_asymptotic_slope",
    "right_asymptotic_slope",
]

In [ ]:
for parameter in RAW_PARAMETER_NAMES:
    fig, axis = plt.subplots(
        figsize=(10, 5)
    )

    axis.plot(
        baseline_parameters[
            "price_date"
        ],
        baseline_parameters[
            parameter
        ],
        marker="o",
        markersize=2,
        label="Independent baseline",
    )

    axis.plot(
        final_parameters[
            "price_date"
        ],
        final_parameters[
            parameter
        ],
        marker="o",
        markersize=2,
        label=(
            "Final ridge "
            f"(lambda={FINAL_RIDGE_LAMBDA:.0e})"
        ),
    )

    axis.set_xlabel(
        "Observation date"
    )

    axis.set_ylabel(
        parameter
    )

    axis.set_title(
        "Baseline versus Final Raw "
        f"SVI Parameter: {parameter}"
    )

    axis.tick_params(
        axis="x",
        rotation=45,
    )

    axis.legend()

    fig.tight_layout()

    fig.savefig(
        PARAMETER_COMPARISON_DIR
        / (
            "raw_parameter_"
            f"{parameter}.png"
        ),
        dpi=230,
        bbox_inches="tight",
    )

    plt.close(fig)

for quantity in DERIVED_QUANTITY_NAMES:
    fig, axis = plt.subplots(
        figsize=(10, 5)
    )

    axis.plot(
        baseline_parameters[
            "price_date"
        ],
        baseline_parameters[
            quantity
        ],
        marker="o",
        markersize=2,
        label="Independent baseline",
    )

    axis.plot(
        final_parameters[
            "price_date"
        ],
        final_parameters[
            quantity
        ],
        marker="o",
        markersize=2,
        label=(
            "Final ridge "
            f"(lambda={FINAL_RIDGE_LAMBDA:.0e})"
        ),
    )

    axis.set_xlabel(
        "Observation date"
    )

    axis.set_ylabel(
        quantity
    )

    axis.set_title(
        "Baseline versus Final Curve "
        f"Quantity: {quantity}"
    )

    axis.tick_params(
        axis="x",
        rotation=45,
    )

    axis.legend()

    fig.tight_layout()

    fig.savefig(
        PARAMETER_COMPARISON_DIR
        / (
            "derived_quantity_"
            f"{quantity}.png"
        ),
        dpi=230,
        bbox_inches="tight",
    )

    plt.close(fig)

print(
    "Raw and derived parameter "
    "comparisons saved."
)

In [ ]:
#per parameter smoothing across lambda
parameter_smoothing_plot = (
    parameter_smoothness
    .pivot(
        index="ridge_lambda",
        columns="parameter",
        values="rms_jump_reduction_percent",
    )
    .loc[
        RIDGE_LAMBDA_GRID,
        SVI_OPT_PARAMETER_NAMES,
    ]
)

lambda_labels = [
    format_lambda(
        ridge_lambda
    )
    for ridge_lambda
    in RIDGE_LAMBDA_GRID
]

lambda_positions = np.arange(
    len(
        RIDGE_LAMBDA_GRID
    )
)

fig, axis = plt.subplots(
    figsize=(10, 6)
)

for parameter in SVI_OPT_PARAMETER_NAMES:
    axis.plot(
        lambda_positions,
        parameter_smoothing_plot[
            parameter
        ],
        marker="o",
        label=parameter,
    )

axis.set_xticks(
    lambda_positions
)

axis.set_xticklabels(
    lambda_labels,
    rotation=45,
)

axis.set_xlabel(
    "Ridge lambda"
)

axis.set_ylabel(
    "Reduction in RMS standardized "
    "daily jumps (%)"
)

axis.set_title(
    "Parameter-Specific Effect of "
    "Temporal Regularization"
)

axis.legend(
    ncol=5
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "parameter_smoothing_by_lambda.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR
    / "parameter_smoothing_by_lambda.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
#final residue diagnostics
daily_residual_summary = (
    final_predictions
    .groupby(
        "price_date",
        as_index=False,
    )
    .agg(
        mean_iv_error_pp=(
            "iv_error_pp",
            "mean",
        ),

        median_iv_error_pp=(
            "iv_error_pp",
            "median",
        ),

        q05_iv_error_pp=(
            "iv_error_pp",
            lambda values:
            values.quantile(0.05),
        ),

        q95_iv_error_pp=(
            "iv_error_pp",
            lambda values:
            values.quantile(0.95),
        ),

        mae_iv_pp=(
            "iv_error_pp",
            lambda values:
            values.abs().mean(),
        ),

        rmse_iv_pp=(
            "iv_error_pp",
            lambda values:
            np.sqrt(
                np.mean(
                    values**2
                )
            ),
        ),
    )
)

daily_residual_summary.to_csv(
    TABLE_DIR
    / "final_daily_residual_summary.csv",
    index=False,
)

In [ ]:
fig, axis = plt.subplots(
    figsize=(10, 6)
)

axis.scatter(
    final_predictions[
        "log_moneyness"
    ],
    final_predictions[
        "iv_error_pp"
    ],
    s=8,
    alpha=0.25,
)

axis.axhline(
    0,
    linestyle="--",
)

axis.set_xlabel(
    "Log-moneyness"
)

axis.set_ylabel(
    "IV residual "
    "(percentage points)"
)

axis.set_title(
    "Final Ridge-SVI Residuals "
    "versus Log-Moneyness"
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "final_residuals_vs_log_moneyness.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    FIGURE_DIR
    / "final_residuals_vs_log_moneyness.pdf",
    bbox_inches="tight",
)

plt.show()

fig, axis = plt.subplots(
    figsize=(9, 5)
)

axis.hist(
    final_predictions[
        "iv_error_pp"
    ],
    bins=80,
)

axis.axvline(
    0,
    linestyle="--",
)

axis.set_xlabel(
    "IV residual "
    "(percentage points)"
)

axis.set_ylabel(
    "Number of option quotes"
)

axis.set_title(
    "Distribution of Final "
    "Ridge-SVI Residuals"
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "final_residual_histogram.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

fig, axis = plt.subplots(
    figsize=(11, 5)
)

axis.fill_between(
    daily_residual_summary[
        "price_date"
    ],
    daily_residual_summary[
        "q05_iv_error_pp"
    ],
    daily_residual_summary[
        "q95_iv_error_pp"
    ],
    alpha=0.25,
    label="5th–95th percentile",
)

axis.plot(
    daily_residual_summary[
        "price_date"
    ],
    daily_residual_summary[
        "mean_iv_error_pp"
    ],
    marker="o",
    markersize=2,
    label="Mean residual",
)

axis.plot(
    daily_residual_summary[
        "price_date"
    ],
    daily_residual_summary[
        "median_iv_error_pp"
    ],
    label="Median residual",
)

axis.axhline(
    0,
    linestyle="--",
)

axis.set_xlabel(
    "Observation date"
)

axis.set_ylabel(
    "IV residual "
    "(percentage points)"
)

axis.set_title(
    "Final Ridge-SVI Residuals "
    "through Time"
)

axis.tick_params(
    axis="x",
    rotation=45,
)

axis.legend()

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "final_residuals_through_time.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

In [ ]:
#fixed expiry surface grids
import matplotlib.dates as mdates
from mpl_toolkits.mplot3d import Axes3D

SURFACE_FIGURE_DIR = (
    FIGURE_DIR
    / "surfaces"
)

SURFACE_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

surface_dates = pd.to_datetime(
    available_dates
)

surface_date_numbers = (
    mdates.date2num(
        surface_dates
    )
)

surface_k_min = (
    model_data[
        "log_moneyness"
    ].quantile(0.02)
)

surface_k_max = (
    model_data[
        "log_moneyness"
    ].quantile(0.98)
)

surface_k_grid = np.linspace(
    surface_k_min,
    surface_k_max,
    80,
)

observed_iv_surface = np.full(
    (
        len(surface_dates),
        len(surface_k_grid),
    ),
    np.nan,
)

final_iv_surface = np.full(
    (
        len(surface_dates),
        len(surface_k_grid),
    ),
    np.nan,
)

for date_index, selected_date in enumerate(
    available_dates
):
    observed_date = (
        final_predictions.loc[
            final_predictions[
                "price_date"
            ] == selected_date
        ]
        .sort_values(
            "log_moneyness"
        )
    )

    observed_k = (
        observed_date[
            "log_moneyness"
        ].to_numpy(
            dtype=float
        )
    )

    observed_iv_pp = (
        100.0
        * observed_date[
            "iv_decimal"
        ].to_numpy(
            dtype=float
        )
    )

    fitted_iv_pp = (
        100.0
        * observed_date[
            "predicted_iv"
        ].to_numpy(
            dtype=float
        )
    )

    inside_observed_range = (
        (
            surface_k_grid
            >= observed_k.min()
        )
        &
        (
            surface_k_grid
            <= observed_k.max()
        )
    )

    observed_iv_surface[
        date_index,
        inside_observed_range,
    ] = np.interp(
        surface_k_grid[
            inside_observed_range
        ],
        observed_k,
        observed_iv_pp,
    )

    final_iv_surface[
        date_index,
        inside_observed_range,
    ] = np.interp(
        surface_k_grid[
            inside_observed_range
        ],
        observed_k,
        fitted_iv_pp,
    )

surface_k_mesh, surface_date_mesh = (
    np.meshgrid(
        surface_k_grid,
        surface_date_numbers,
    )
)

print(
    "Surface grids constructed."
)

In [ ]:
fig = plt.figure(
    figsize=(12, 8)
)

axis = fig.add_subplot(
    111,
    projection="3d",
)

axis.plot_surface(
    surface_k_mesh,
    surface_date_mesh,
    observed_iv_surface,
    linewidth=0,
    antialiased=True,
)

axis.set_xlabel(
    "Log-moneyness"
)

axis.set_ylabel(
    "Observation date"
)

axis.set_zlabel(
    "Market implied volatility (%)"
)

axis.set_title(
    "Observed Fixed-Expiry "
    "Implied-Volatility Surface"
)

axis.yaxis.set_major_formatter(
    mdates.DateFormatter(
        "%Y-%m"
    )
)

fig.tight_layout()

fig.savefig(
    SURFACE_FIGURE_DIR
    / "observed_fixed_expiry_iv_surface.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    SURFACE_FIGURE_DIR
    / "observed_fixed_expiry_iv_surface.pdf",
    bbox_inches="tight",
)

plt.show()

fig = plt.figure(
    figsize=(12, 8)
)

axis = fig.add_subplot(
    111,
    projection="3d",
)

axis.plot_surface(
    surface_k_mesh,
    surface_date_mesh,
    final_iv_surface,
    linewidth=0,
    antialiased=True,
)

axis.set_xlabel(
    "Log-moneyness"
)

axis.set_ylabel(
    "Observation date"
)

axis.set_zlabel(
    "Fitted implied volatility (%)"
)

axis.set_title(
    "Final Ridge-SVI Fixed-Expiry "
    "Implied-Volatility Surface"
)

axis.yaxis.set_major_formatter(
    mdates.DateFormatter(
        "%Y-%m"
    )
)

fig.tight_layout()

fig.savefig(
    SURFACE_FIGURE_DIR
    / "final_ridge_svi_iv_surface.png",
    dpi=250,
    bbox_inches="tight",
)

fig.savefig(
    SURFACE_FIGURE_DIR
    / "final_ridge_svi_iv_surface.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
final_standardized_change_columns = [
    f"standardized_change_{parameter}"
    for parameter
    in SVI_OPT_PARAMETER_NAMES
]

final_parameter_jump_records = []

for parameter in SVI_OPT_PARAMETER_NAMES:
    changes = (
        final_parameters[
            f"standardized_change_{parameter}"
        ]
        .iloc[1:]
        .to_numpy(
            dtype=float
        )
    )

    final_parameter_jump_records.append(
        {
            "parameter": parameter,

            "rms_standardized_jump": (
                np.sqrt(
                    np.mean(
                        changes**2
                    )
                )
            ),

            "mean_absolute_standardized_jump": (
                np.mean(
                    np.abs(
                        changes
                    )
                )
            ),

            "maximum_absolute_standardized_jump": (
                np.max(
                    np.abs(
                        changes
                    )
                )
            ),
        }
    )

final_parameter_jump_summary = (
    pd.DataFrame(
        final_parameter_jump_records
    )
    .set_index(
        "parameter"
    )
)

display(
    final_parameter_jump_summary
)

lambda_diagnostic_summary = (
    regularized_all_parameters
    .groupby(
        "ridge_lambda",
        as_index=False,
    )
    .agg(
        n_dates=(
            "price_date",
            "size",
        ),

        optimizer_successes=(
            "optimizer_success",
            "sum",
        ),

        dates_with_box_bound_hits=(
            "n_box_bound_hits",
            lambda values:
            int(
                (
                    values > 0
                ).sum()
            ),
        ),

        dates_with_active_constraints=(
            "n_active_constraints",
            lambda values:
            int(
                (
                    values > 0
                ).sum()
            ),
        ),

        mean_number_successful_starts=(
            "n_successful_starts",
            "mean",
        ),

        minimum_number_valid_starts=(
            "n_valid_starts",
            "min",
        ),
    )
)

display(
    lambda_diagnostic_summary
)

daily_error_delta = (
    baseline_daily_export[
        [
            "price_date",
            "rmse_iv_percentage_points",
            "weighted_mse_total_variance",
        ]
    ]
    .rename(
        columns={
            "rmse_iv_percentage_points":
            "baseline_iv_rmse_pp",

            "weighted_mse_total_variance":
            "baseline_weighted_mse",
        }
    )
    .merge(
        final_daily_metrics[
            [
                "price_date",
                "rmse_iv_percentage_points",
                "weighted_mse_total_variance",
            ]
        ]
        .rename(
            columns={
                "rmse_iv_percentage_points":
                "final_iv_rmse_pp",

                "weighted_mse_total_variance":
                "final_weighted_mse",
            }
        ),
        on="price_date",
        how="inner",
        validate="one_to_one",
    )
)

daily_error_delta[
    "iv_rmse_change_pp"
] = (
    daily_error_delta[
        "final_iv_rmse_pp"
    ]
    -
    daily_error_delta[
        "baseline_iv_rmse_pp"
    ]
)

daily_error_delta[
    "weighted_mse_change"
] = (
    daily_error_delta[
        "final_weighted_mse"
    ]
    -
    daily_error_delta[
        "baseline_weighted_mse"
    ]
)

final_parameter_jump_summary.to_csv(
    TABLE_DIR
    / "final_parameter_jump_summary.csv"
)

lambda_diagnostic_summary.to_csv(
    TABLE_DIR
    / "lambda_optimizer_constraint_diagnostics.csv",
    index=False,
)

daily_error_delta.to_csv(
    TABLE_DIR
    / "baseline_vs_final_daily_error_changes.csv",
    index=False,
)

residual_heatmap_summary.to_csv(
    TABLE_DIR
    / "final_residual_heatmap_data.csv",
    index=False,
)

print(
    "Final diagnostic tables saved."
)

In [ ]:
#output summary
selected_lambda_results = (
    lambda_summary.loc[
        lambda_summary[
            "ridge_lambda"
        ] == FINAL_RIDGE_LAMBDA
    ]
    .iloc[0]
)

baseline_overall_row = (
    overall_model_comparison.loc[
        overall_model_comparison[
            "model"
        ] == "Independent baseline"
    ]
    .iloc[0]
)

final_overall_row = (
    overall_model_comparison.loc[
        overall_model_comparison[
            "model"
        ].str.startswith(
            "Final ridge"
        )
    ]
    .iloc[0]
)

In [ ]:
#completed output directory
def print_directory_tree(
    directory,
    prefix="",
):
    entries = sorted(
        directory.iterdir(),
        key=lambda path: (
            not path.is_dir(),
            path.name.lower(),
        ),
    )

    for index, entry in enumerate(
        entries
    ):
        is_last = (
            index
            == len(entries) - 1
        )

        connector = (
            "└── "
            if is_last
            else "├── "
        )

        print(
            prefix
            + connector
            + entry.name
        )

        if entry.is_dir():
            extension = (
                "    "
                if is_last
                else "│   "
            )

            print_directory_tree(
                entry,
                prefix + extension,
            )


print(
    OUTPUT_DIR.name
    + "/"
)

print_directory_tree(
    OUTPUT_DIR
)